# Financial Time Series and Their Characteristics
## A Guided Python Workbook on Chapter 1 of Tsay (2010)

**ACCFIN5277 Finance Programming in Python**<br>
Adam Smith Business School, University of Glasgow

**Prepared by:** Sadra Heydari<br>
**Textbook:** Tsay, R. S. (2010). *Analysis of Financial Time Series*, 3rd edition. Hoboken, NJ: Wiley. Chapter 1.

---

### About this workbook

This workbook turns the formulas of Chapter 1 into Python, one step at a time. In Part A you write every function yourself using only basic Python, so that you see exactly how each formula is computed. In Part B you meet the libraries that professionals use (NumPy, SciPy, pandas and Matplotlib) and check that they give the same answers as your own code.

### Learning outcomes

After completing the workbook you should be able to:

1. compute simple, logarithmic, multiperiod, annualised, portfolio and excess returns;
2. explain and compute compounding, continuous compounding and present value;
3. estimate the sample mean, variance, skewness and kurtosis of a return series;
4. test whether returns have zero mean, zero skewness, zero excess kurtosis and a normal distribution;
5. describe the normal, lognormal, stable (Cauchy) and mixture-of-normals models for returns;
6. reproduce results from the chapter (Table 1.1, Example 1.2, Table 1.2) and solve Exercises 1.1 to 1.5;
7. use NumPy, SciPy and pandas for the same calculations and recognise where their definitions differ from the textbook.

### Contents

* **Part 0. Getting started**: setup, Python essentials, loading the data
* **Part A. Building the toolkit yourself**
  * 1.1 Asset returns
  * 1.2.1 Moments and tests of normality
  * 1.2.2 Distributions of returns
  * 1.2.3 Multivariate returns
  * 1.2.4 The likelihood function
  * 1.2.5 Empirical properties of returns
  * 1.3 Exercises of Chapter 1
* **Part B. Letting the libraries do the work**
* **Summary and references**

Section numbers follow the textbook, so you can read the book and this workbook side by side.

### How to use this workbook

Each **Task** has the same structure:

1. **Read** the short explanation and the formula.
2. **Write your code** in the task cell, replacing each line `raise NotImplementedError(...)`. Then run the cell (Shift + Enter).
3. **Run the test cell** below it. Each line reports `✓ PASS`, `✗ FAIL` (with the expected answer) or `… TODO` (not written yet).
4. If you are stuck, open **Show hint**. Open **Show solution** only after a genuine attempt.

**Questions** ask you to interpret a result; think of your answer before you open it.

**Rules for Part A.** Use plain Python only: numbers, lists, `for` loops, `if` statements, functions, and the built-in `math` module (`math.log`, `math.exp`, `math.pi`, `math.erf`). The built-in functions `len`, `abs`, `min` and `max` are allowed. NumPy, SciPy and pandas are not used until Part B.

### Notation

The code uses the same symbols as the book. Python is case-sensitive, so `R` and `r` are different names.

| Book | Python | Meaning |
|---|---|---|
| $P_t$, $P_{t-1}$ | `P_t`, `P_t_minus_1`; a list `P` with elements `P[t]` | price at time $t$ and $t-1$ |
| $R_t$, $1+R_t[k]$ | `R_t`; a list `R` | simple return; $k$-period gross return |
| $r_t = \ln(1+R_t)$ | `r_t`; a list `r` | log (continuously compounded) return |
| $k$, $T$, $N$ | `k`, `T`, `N` | number of periods, sample size, number of assets |
| $C$, $A$, $r$, $m$, $n$ | `C`, `A`, `r`, `m`, `n` | initial capital, final value, annual interest rate, payments per year, years |
| $w_i$, $R_{p,t}$ | `w[i]`, `R_p` | portfolio weight, portfolio return |
| $D_t$; $R_{0t}$, $Z_t$ | `D_t`; `R0`, `Z` | dividend; reference return, excess return |
| $x_1, \dots, x_T$ | a list `x`, element `x_t` | a sample of observations |
| $\hat\mu_x$, $\hat\sigma_x^2$, $\hat\sigma_x$ | `mu_hat`, `sigma2_hat`, `sigma_hat` | sample mean, variance, standard deviation |
| $\hat S(x)$, $\hat K(x)$ | `S_hat`, `K_hat` | sample skewness and kurtosis |
| $\mu$, $\sigma^2$, $\alpha$ | `mu`, `sigma2`, `alpha` | model parameters |
| $\hat{\boldsymbol\mu}_x$, $\hat{\boldsymbol\Sigma}_x$ | `mu_hat`, `Sigma_hat` | sample mean vector, covariance matrix |

---
# Part 0. Getting started

### 0.1 Setup

Run the two cells below once at the start of every session. The first defines `load_data`, which downloads a data file from the textbook's website; the second defines the testing tools `check` and `show_progress`.

In [ ]:
import math
import os
import urllib.request

DATA_URL = "https://faculty.chicagobooth.edu/-/media/faculty/ruey-s-tsay/teaching/fts3/"


def load_data(filename):
    """Read one of the book's data files and return its columns.

    Example:  da = load_data("d-ibm3dx7008.txt")
              da["rtn"]  ->  list of IBM daily returns
    The file is downloaded from the book's website. If a copy with the same
    name is saved next to this notebook, that copy is used instead.
    """
    if os.path.exists(filename):
        text = open(filename).read()
    else:
        request = urllib.request.Request(DATA_URL + filename, headers={"User-Agent": "Mozilla/5.0"})
        text = urllib.request.urlopen(request).read().decode("utf-8-sig")

    lines = text.strip().split("\n")
    names = lines[0].lower().split()          # column names from the first line
    data = {}
    for name in names:
        data[name] = []
    for line in lines[1:]:
        values = line.split()
        for i in range(len(names)):
            data[names[i]].append(float(values[i]))
    return data


print("load_data() is ready.")

In [ ]:
# Testing tools used throughout the workbook. You do not need to read this cell.
_results = {}


def _close(a, b, tol):
    if isinstance(b, dict):
        return isinstance(a, dict) and all(k in a and _close(a[k], b[k], tol) for k in b)
    if isinstance(b, (list, tuple)):
        return (isinstance(a, (list, tuple)) and len(a) == len(b)
                and all(_close(a[i], b[i], tol) for i in range(len(b))))
    try:
        return abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))
    except (TypeError, ValueError):
        return False


def _show(v):
    if isinstance(v, (list, tuple)) and len(v) > 10:
        return f"<list of {len(v)} numbers>"
    return repr(v)


def _report(status, text):
    _results[text] = status
    print({"PASS": "✓ PASS", "FAIL": "✗ FAIL", "TODO": "… TODO"}[status], " ", text)


def check(function, *inputs, expected, tol=1e-6, label=""):
    """Call function(*inputs) and compare the result with the expected answer."""
    text = function.__name__ + "(" + ", ".join(_show(v) for v in inputs) + ")"
    if label:
        text += "   [" + label + "]"
    if any(v is None for v in inputs):
        return _report("TODO", text + "   <- the input has not been computed yet")
    try:
        result = function(*inputs)
    except NotImplementedError:
        return _report("TODO", text + "   <- not written yet")
    except Exception as error:
        return _report("FAIL", text + f"   <- error: {type(error).__name__}: {error}")
    if _close(result, expected, tol):
        _report("PASS", text)
    else:
        _report("FAIL", text + f"   <- expected {expected}, got {_show(result)}")


def check_value(label, value, expected, tol=1e-6):
    """Compare a value you computed with the expected answer."""
    if value is None:
        _report("TODO", label + "   <- still None")
    elif _close(value, expected, tol):
        _report("PASS", label)
    else:
        _report("FAIL", label + f"   <- expected {expected}, got {_show(value)}")


def show_progress():
    """Count the tests run so far in this session."""
    n_pass = list(_results.values()).count("PASS")
    n_fail = list(_results.values()).count("FAIL")
    n_todo = list(_results.values()).count("TODO")
    print(f"{n_pass} passed, {n_fail} failed, {n_todo} still to do")
    for text, status in _results.items():
        if status == "FAIL":
            print("  still failing:", text)


print("check() and show_progress() are ready.")

### 0.2 Python essentials

If you are new to Python, run the following cells and read the comments. They contain everything you need for Part A.

**Numbers, variables and arithmetic.** `**` means "to the power of", so `2 ** 3` is 8 and `x ** 0.5` is the square root of `x`.

In [ ]:
P_t_minus_1 = 100.0          # price yesterday
P_t = 105.0                  # price today
R_t = P_t / P_t_minus_1 - 1  # simple return
print("R_t =", R_t)
print("square root of 16 =", 16 ** 0.5)

**Lists and loops.** A list holds many numbers. Positions start at 0, so `P[0]` is the first element and `P[-1]` the last. A `for` loop visits the elements one by one; a variable such as `total` that is updated inside the loop is called an *accumulator*.

In [ ]:
P = [100.0, 110.0, 99.0, 120.0]
print("number of prices:", len(P))
print("first price:", P[0], "  last price:", P[-1])

total = 0.0
for P_t in P:
    total = total + P_t
print("sum of the prices:", total)

# range(1, len(P)) gives t = 1, 2, 3, so we can use P[t] and P[t - 1] together
R = []
for t in range(1, len(P)):
    R_t = P[t] / P[t - 1] - 1
    R.append(R_t)              # append adds an element to the end of a list
print("returns:", R)

**Functions and the `math` module.** A function packages a calculation: it receives inputs and sends back a result with `return`.

In [ ]:
def gross_return(R_t):
    return 1 + R_t


print("gross return:", gross_return(0.05))
print("ln(1.05) =", math.log(1.05), "  exp(0.05) =", math.exp(0.05), "  pi =", math.pi)

### 0.3 Loading the data

The data files used in the chapter are on the textbook's website. `load_data` downloads a file and returns a *dictionary*: a collection of columns, each looked up by its name. Compare with the R command in the book, `da=read.table("d-ibm3dx7008.txt",header=T)`.

In [ ]:
da = load_data("d-ibm3dx7008.txt")
print("columns:", list(da.keys()))
print("number of days:", len(da["rtn"]))
print("first five IBM returns:", da["rtn"][:5])

`da["rtn"]` is the list of daily **simple returns** of IBM stock from 1970 to 2008 (the other columns are the CRSP value-weighted index `vwretd`, the equal-weighted index `ewretd` and the S&P composite `sprtrn`). Returns are stored as decimals: 0.01 means 1%.

The files used in this workbook are:

| File | Content | Used in |
|---|---|---|
| `d-ibm3dx7008.txt` | daily simple returns of IBM, VW, EW and S&P, 1970 to 2008 | Example 1.2, Table 1.2 |
| `d-3stocks9908.txt` | daily simple returns of AXP, CAT and SBUX, 1999 to 2008 | Exercises 1.1, 1.4 |
| `m-gm3dx7508.txt` | monthly simple returns of GM, VW, EW and S&P, 1975 to 2008 | Exercises 1.2, 1.3 |
| `d-caus.txt`, `d-useu.txt`, `d-jpus.txt`, `d-usuk.txt` | daily exchange rates, January 2000 to March 2009 | Exercise 1.5 |

---
# Part A. Building the toolkit yourself

## 1.1 Asset Returns

Most financial studies work with returns rather than prices, for two reasons (Campbell, Lo and MacKinlay, 1997): a return is a complete and scale-free summary of an investment opportunity, and return series have more attractive statistical properties than price series. Throughout, $P_t$ denotes the price of an asset at time $t$, and we assume for now that the asset pays no dividends.

### One-period simple return

> **Definition.** Holding an asset from date $t-1$ to date $t$ gives the **simple gross return**
> $$1 + R_t = \frac{P_t}{P_{t-1}} \qquad\text{or}\qquad P_t = P_{t-1}(1 + R_t), \tag{1.1}$$
> and the **simple net return** (or simple return)
> $$R_t = \frac{P_t}{P_{t-1}} - 1 = \frac{P_t - P_{t-1}}{P_{t-1}}. \tag{1.2}$$

**Task 1.** Write `simple_gross_return(P_t_minus_1, P_t)` and `simple_return(P_t_minus_1, P_t)`, implementing Eqs. (1.1) and (1.2).

In [ ]:
# Task 1: write your code in place of each `raise NotImplementedError(...)` line
def simple_gross_return(P_t_minus_1, P_t):
    """Simple gross return 1 + R_t, Eq. (1.1)."""
    raise NotImplementedError("Task 1 is not completed yet")


def simple_return(P_t_minus_1, P_t):
    """Simple net return R_t, Eq. (1.2)."""
    raise NotImplementedError("Task 1 is not completed yet")

In [ ]:
# Tests for Task 1: run this cell. Every line should say PASS.
check(simple_gross_return, 100, 105, expected=1.05)
check(simple_gross_return, 50, 40, expected=0.8)
check(simple_return, 100, 105, expected=0.05, label='a 5% gain')
check(simple_return, 50, 40, expected=-0.2, label='a 20% loss')
check(simple_return, 80, 80, expected=0.0, label='no change')

<details>
<summary><b>Show solution to Task 1</b></summary>

```python
def simple_gross_return(P_t_minus_1, P_t):
    """Simple gross return 1 + R_t, Eq. (1.1)."""
    return P_t / P_t_minus_1


def simple_return(P_t_minus_1, P_t):
    """Simple net return R_t, Eq. (1.2)."""
    R_t = P_t / P_t_minus_1 - 1
    return R_t
```

</details>

In practice we have a whole price series $P_0, P_1, \dots, P_T$ and want the return series $R_1, \dots, R_T$. Note that there is one return fewer than there are prices.

**Task 2.** Write `simple_returns(P)`, which takes a list of prices `P` and returns the list of one-period simple returns. Use your `simple_return` inside a loop.

In [ ]:
# Task 2: write your code in place of each `raise NotImplementedError(...)` line
def simple_returns(P):
    """List of simple returns [R_1, ..., R_T] from a list of prices [P_0, ..., P_T]."""
    raise NotImplementedError("Task 2 is not completed yet")

In [ ]:
# Tests for Task 2: run this cell. Every line should say PASS.
check(simple_returns, [100, 110, 99, 99], expected=[0.1, -0.1, 0.0])
check(simple_returns, [20, 25], expected=[0.25], label='two prices give one return')
check(simple_returns, [1, 2, 4, 8, 16], expected=[1.0, 1.0, 1.0, 1.0], label='doubling every period')

<details>
<summary><b>Show hint</b></summary>

Start with an empty list `R = []`. Loop over `t` in `range(1, len(P))`; for each `t` compute the return from `P[t - 1]` to `P[t]` and `append` it to `R`. Section 0.2 has an almost identical example.

</details>

<details>
<summary><b>Show solution to Task 2</b></summary>

```python
def simple_returns(P):
    """List of simple returns [R_1, ..., R_T] from a list of prices [P_0, ..., P_T]."""
    R = []
    for t in range(1, len(P)):
        R_t = simple_return(P[t - 1], P[t])
        R.append(R_t)
    return R
```

</details>

### Multiperiod simple return

> **Definition.** Holding the asset for $k$ periods, from $t-k$ to $t$, gives the **$k$-period simple gross return**
> $$1 + R_t[k] = \frac{P_t}{P_{t-k}} = \frac{P_t}{P_{t-1}} \times \frac{P_{t-1}}{P_{t-2}} \times \cdots \times \frac{P_{t-k+1}}{P_{t-k}} = \prod_{j=0}^{k-1}(1 + R_{t-j}).$$

The $k$-period gross return is the *product* of the one-period gross returns, which is why it is called a *compound* return.

**Task 3.** Write `multiperiod_gross_return(R)`, which multiplies the one-period gross returns $(1 + R_t)$ in the list `R`.

In [ ]:
# Task 3: write your code in place of each `raise NotImplementedError(...)` line
def multiperiod_gross_return(R):
    """k-period gross return 1 + R_t[k] from a list R of k one-period simple returns."""
    raise NotImplementedError("Task 3 is not completed yet")

In [ ]:
# Tests for Task 3: run this cell. Every line should say PASS.
check(multiperiod_gross_return, [0.1, -0.1], expected=0.99, label='+10% then -10%')
check(multiperiod_gross_return, [0.05, 0.05, 0.05], expected=1.157625, label='5% for three periods')
check(multiperiod_gross_return, [0.1, -0.1, 0.2121212121], expected=1.2, label='equals P_t / P_(t-k) = 120/100')

<details>
<summary><b>Show hint</b></summary>

This is an accumulator like `total` in Section 0.2, but for a product: start from `gross = 1.0` and multiply by `(1 + R_t)` inside the loop.

</details>

<details>
<summary><b>Show solution to Task 3</b></summary>

```python
def multiperiod_gross_return(R):
    """k-period gross return 1 + R_t[k] from a list R of k one-period simple returns."""
    gross = 1.0
    for R_t in R:
        gross = gross * (1 + R_t)
    return gross
```

</details>

**Question 1.** The first test shows that a gain of 10% followed by a loss of 10% gives a gross return of 0.99, not 1. Why is this not break-even?

<details>
<summary><b>Show answer</b></summary>

The loss is taken from a larger base. From 100 the price rises to 110, and 10% of 110 is 11, so it falls to 99. Gross returns multiply ($1.1 \times 0.9 = 0.99$); adding the net returns ($+10\% - 10\% = 0$) ignores compounding.

</details>

### Annualised return

If the asset was held for $k$ years, the **annualised (average) return** is the geometric mean of the $k$ annual gross returns, minus one:
$$\text{Annualized}\{R_t[k]\} = \Big[\prod_{j=0}^{k-1}(1+R_{t-j})\Big]^{1/k} - 1 .$$
Because one-period returns tend to be small, a first-order Taylor expansion gives the simpler approximation
$$\text{Annualized}\{R_t[k]\} \approx \frac{1}{k}\sum_{j=0}^{k-1} R_{t-j}, \tag{1.3}$$
which is just the arithmetic mean of the returns.

**Task 4.** Write `annualized_return(R)` (exact, geometric mean) and `annualized_return_approx(R)` (Eq. 1.3, arithmetic mean). Reuse `multiperiod_gross_return` in the first one.

In [ ]:
# Task 4: write your code in place of each `raise NotImplementedError(...)` line
def annualized_return(R):
    """Exact annualised return: [product of (1 + R_t)] ** (1 / k) - 1."""
    raise NotImplementedError("Task 4 is not completed yet")


def annualized_return_approx(R):
    """Approximate annualised return, Eq. (1.3): the average of the returns."""
    raise NotImplementedError("Task 4 is not completed yet")

In [ ]:
# Tests for Task 4: run this cell. Every line should say PASS.
check(annualized_return, [0.2, 0.2, 0.2], expected=0.2, label='20% every year')
check(annualized_return, [0.1, -0.1], expected=-0.005012562893)
check(annualized_return, [0.5, -0.5], expected=-0.1339745962)
check(annualized_return_approx, [0.01, 0.02, 0.03], expected=0.02)
check(annualized_return_approx, [0.5, -0.5], expected=0.0)

<details>
<summary><b>Show hint</b></summary>

The $k$-th root of a number `g` is `g ** (1 / k)`. For the approximation, add up the returns with an accumulator and divide by `k = len(R)`.

</details>

<details>
<summary><b>Show solution to Task 4</b></summary>

```python
def annualized_return(R):
    """Exact annualised return: [product of (1 + R_t)] ** (1 / k) - 1."""
    k = len(R)
    gross = multiperiod_gross_return(R)
    return gross ** (1 / k) - 1


def annualized_return_approx(R):
    """Approximate annualised return, Eq. (1.3): the average of the returns."""
    k = len(R)
    total = 0.0
    for R_t in R:
        total = total + R_t
    return total / k
```

</details>

**Question 2.** For the returns +50% and −50%, Eq. (1.3) gives an average of 0%, whereas the exact annualised return is about −13.4%. When is the approximation (1.3) poor?

<details>
<summary><b>Show answer</b></summary>

The investor lost 25% over two years ($1.5 \times 0.5 = 0.75$), i.e. about −13.4% per year. The arithmetic mean ignores compounding, and the error grows with the **volatility** of the returns: roughly, the geometric mean equals the arithmetic mean minus half the variance. For small daily or monthly returns (1.3) is accurate; for large swings it is not.

</details>

### Continuous compounding

Suppose a bank pays interest at the rate $r = 10\%$ per annum on an initial deposit $C$. If interest is paid $m$ times a year, each payment is at the rate $r/m$ and after $n$ years the deposit is worth $C(1 + r/m)^{mn}$. Table 1.1 of the book shows that, as $m$ increases, the value after one year approaches a limit, $C e^{r}$. This limit is **continuous compounding**:

> $$A = C \exp(r \times n), \tag{1.4}$$
> $$C = A \exp(-r \times n). \tag{1.5}$$

Equation (1.5) gives the **present value** $C$ of an amount $A$ received $n$ years from now.

**Task 5.** Write `discrete_compound_value(C, r, m, n)`, `continuous_compound_value(C, r, n)` (Eq. 1.4) and `present_value(A, r, n)` (Eq. 1.5).

In [ ]:
# Task 5: write your code in place of each `raise NotImplementedError(...)` line
def discrete_compound_value(C, r, m, n):
    """Value after n years of C invested at annual rate r, paid m times a year: C (1 + r/m)^(m n)."""
    raise NotImplementedError("Task 5 is not completed yet")


def continuous_compound_value(C, r, n):
    """Eq. (1.4): A = C exp(r n)."""
    raise NotImplementedError("Task 5 is not completed yet")


def present_value(A, r, n):
    """Eq. (1.5): C = A exp(-r n)."""
    raise NotImplementedError("Task 5 is not completed yet")

In [ ]:
# Tests for Task 5: run this cell. Every line should say PASS.
check(discrete_compound_value, 1, 0.10, 1, 1, expected=1.1, label='annual payment')
check(discrete_compound_value, 1, 0.10, 2, 1, expected=1.1025, label='semiannual payments')
check(discrete_compound_value, 1, 0.10, 1, 2, expected=1.21, label='annual payments for two years')
check(continuous_compound_value, 1, 0.10, 1, expected=1.105170918)
check(continuous_compound_value, 100, 0.05, 10, expected=164.8721271)
check(present_value, 1, 0.10, 1, expected=0.904837418)
check(present_value, 164.8721271, 0.05, 10, expected=100.0, label='undoes the previous calculation')

<details>
<summary><b>Show solution to Task 5</b></summary>

```python
def discrete_compound_value(C, r, m, n):
    """Value after n years of C invested at annual rate r, paid m times a year: C (1 + r/m)^(m n)."""
    A = C * (1 + r / m) ** (m * n)
    return A


def continuous_compound_value(C, r, n):
    """Eq. (1.4): A = C exp(r n)."""
    A = C * math.exp(r * n)
    return A


def present_value(A, r, n):
    """Eq. (1.5): C = A exp(-r n)."""
    C = A * math.exp(-r * n)
    return C
```

</details>

**Table 1.1 of the book.** With your functions, the next cell reproduces the value of \$1.00 after one year at 10% per annum for different payment frequencies.

In [ ]:
names = ["Annual", "Semiannual", "Quarterly", "Monthly", "Weekly", "Daily"]
payments = [1, 2, 4, 12, 52, 365]

print("Type            m    Net value")
for i in range(len(payments)):
    A = discrete_compound_value(1.0, 0.10, payments[i], 1)
    print(f"{names[i]:<12} {payments[i]:>5}    {A:.5f}")
print(f"{'Continuously':<12} {'inf':>5}    {continuous_compound_value(1.0, 0.10, 1):.5f}")

### Continuously compounded return

> **Definition.** The natural logarithm of the simple gross return is the **continuously compounded return** or **log return**:
> $$r_t = \ln(1 + R_t) = \ln\frac{P_t}{P_{t-1}} = p_t - p_{t-1}, \qquad p_t = \ln(P_t). \tag{1.6}$$

Log returns have a very convenient property: the $k$-period log return is the **sum** of the one-period log returns,
$$r_t[k] = \ln(1 + R_t[k]) = r_t + r_{t-1} + \cdots + r_{t-k+1}.$$

**Task 6.** Write `log_return(P_t_minus_1, P_t)` (Eq. 1.6), `log_returns(P)` for a list of prices, and `multiperiod_log_return(r)`, which adds up a list of log returns.

In [ ]:
# Task 6: write your code in place of each `raise NotImplementedError(...)` line
def log_return(P_t_minus_1, P_t):
    """Log return r_t, Eq. (1.6)."""
    raise NotImplementedError("Task 6 is not completed yet")


def log_returns(P):
    """List of log returns [r_1, ..., r_T] from a list of prices [P_0, ..., P_T]."""
    raise NotImplementedError("Task 6 is not completed yet")


def multiperiod_log_return(r):
    """k-period log return r_t[k]: the sum of the one-period log returns in the list r."""
    raise NotImplementedError("Task 6 is not completed yet")

In [ ]:
# Tests for Task 6: run this cell. Every line should say PASS.
check(log_return, 100, 110, expected=0.0953101798)
check(log_return, 100, 100, expected=0.0, label='no change')
check(log_returns, [100, 110, 99], expected=[0.0953101798, -0.1053605157])
check(multiperiod_log_return, [0.0953101798, -0.1053605157, 0.1923718926], expected=0.1823215567, label='equals ln(120/100)')
check(multiperiod_log_return, [4.46, -7.34, 10.77], expected=7.89, label='Example 1.1: quarterly log return in %')

<details>
<summary><b>Show solution to Task 6</b></summary>

```python
def log_return(P_t_minus_1, P_t):
    """Log return r_t, Eq. (1.6)."""
    r_t = math.log(P_t / P_t_minus_1)
    return r_t


def log_returns(P):
    """List of log returns [r_1, ..., r_T] from a list of prices [P_0, ..., P_T]."""
    r = []
    for t in range(1, len(P)):
        r_t = log_return(P[t - 1], P[t])
        r.append(r_t)
    return r


def multiperiod_log_return(r):
    """k-period log return r_t[k]: the sum of the one-period log returns in the list r."""
    total = 0.0
    for r_t in r:
        total = total + r_t
    return total
```

</details>

### Relationship between simple and log returns

The two kinds of return are linked by
$$r_t = \ln(1 + R_t), \qquad R_t = e^{r_t} - 1 .$$
When returns are given in percent, divide by 100 before converting and multiply by 100 afterwards: $r_t = 100\ln(1 + R_t/100)$ and $R_t = 100\,(e^{r_t/100} - 1)$.

**Task 7.** Write `simple_to_log(R_t)` and `log_to_simple(r_t)`. Both work with returns in decimals.

In [ ]:
# Task 7: write your code in place of each `raise NotImplementedError(...)` line
def simple_to_log(R_t):
    """Log return r_t = ln(1 + R_t)."""
    raise NotImplementedError("Task 7 is not completed yet")


def log_to_simple(r_t):
    """Simple return R_t = exp(r_t) - 1."""
    raise NotImplementedError("Task 7 is not completed yet")

In [ ]:
# Tests for Task 7: run this cell. Every line should say PASS.
check(simple_to_log, 0.05, expected=0.04879016417)
check(log_to_simple, 0.0487901642, expected=0.05000000003, label='back to 5%')
check(log_to_simple, 0.0446, expected=0.04560953244, label='Example 1.1: 4.46% log return = 4.56% simple')
check(simple_to_log, -0.5, expected=-0.6931471806, label='a 50% loss')

<details>
<summary><b>Show solution to Task 7</b></summary>

```python
def simple_to_log(R_t):
    """Log return r_t = ln(1 + R_t)."""
    r_t = math.log(1 + R_t)
    return r_t


def log_to_simple(r_t):
    """Simple return R_t = exp(r_t) - 1."""
    R_t = math.exp(r_t) - 1
    return R_t
```

</details>

**Example 1.1 of the book.** A monthly log return of 4.46% corresponds to a simple return of $100\,[\exp(4.46/100) - 1] = 4.56\%$. If the monthly log returns within a quarter are 4.46%, −7.34% and 10.77%, the quarterly log return is their sum, 7.89%.

In [ ]:
R_t = 100 * log_to_simple(4.46 / 100)
print("simple return (%):", round(R_t, 2))

r_quarter = multiperiod_log_return([4.46, -7.34, 10.77])
print("quarterly log return (%):", round(r_quarter, 2))

**Question 3.** For any $R_t \neq 0$ the log return $\ln(1+R_t)$ is smaller than the simple return $R_t$. Why? What is the smallest possible value of each?

<details>
<summary><b>Show answer</b></summary>

Because $\ln(1+x) \le x$ for every $x > -1$, with equality only at $x = 0$ (the logarithm is concave). The gap is tiny for small returns and large for big moves. A simple return can never be below −100%, since a price cannot become negative, whereas a log return has no lower bound: $\ln(1+R_t) \to -\infty$ as $R_t \to -1$.

</details>

### Portfolio return

> **Definition.** For a portfolio of $N$ assets with weights $w_i$ (the fraction of the portfolio's value invested in asset $i$), the simple return is the weighted average of the assets' simple returns:
> $$R_{p,t} = \sum_{i=1}^{N} w_i R_{it}.$$

Log returns do not have this exact property, but when returns are small, $r_{p,t} \approx \sum_{i=1}^N w_i r_{it}$.

**Task 8.** Write `portfolio_simple_return(w, R)`, where `w` is the list of weights and `R` the list of the assets' simple returns in the same period.

In [ ]:
# Task 8: write your code in place of each `raise NotImplementedError(...)` line
def portfolio_simple_return(w, R):
    """Portfolio simple return R_p = w_1 R_1 + ... + w_N R_N."""
    raise NotImplementedError("Task 8 is not completed yet")

In [ ]:
# Tests for Task 8: run this cell. Every line should say PASS.
check(portfolio_simple_return, [0.5, 0.5], [0.10, -0.02], expected=0.04, label='equal weights')
check(portfolio_simple_return, [0.2, 0.3, 0.5], [0.01, 0.02, 0.03], expected=0.023)
check(portfolio_simple_return, [1.5, -0.5], [0.04, 0.02], expected=0.05, label='150% long, 50% short')

<details>
<summary><b>Show hint</b></summary>

Loop over the positions `i` in `range(N)` and add `w[i] * R[i]` to an accumulator.

</details>

<details>
<summary><b>Show solution to Task 8</b></summary>

```python
def portfolio_simple_return(w, R):
    """Portfolio simple return R_p = w_1 R_1 + ... + w_N R_N."""
    N = len(w)
    R_p = 0.0
    for i in range(N):
        R_p = R_p + w[i] * R[i]
    return R_p
```

</details>

How good is the approximation for log returns? The next cell compares the exact portfolio log return with the weighted average of the log returns, first for small and then for large returns.

In [ ]:
w = [0.6, 0.4]

R_small = [0.02, -0.01]
exact = simple_to_log(portfolio_simple_return(w, R_small))
approx = w[0] * simple_to_log(R_small[0]) + w[1] * simple_to_log(R_small[1])
print("small returns:  exact =", round(exact, 5), "  approximation =", round(approx, 5))

R_large = [0.40, -0.30]
exact = simple_to_log(portfolio_simple_return(w, R_large))
approx = w[0] * simple_to_log(R_large[0]) + w[1] * simple_to_log(R_large[1])
print("large returns:  exact =", round(exact, 5), "  approximation =", round(approx, 5))

### Dividend payment

If the asset pays a dividend $D_t$ between dates $t-1$ and $t$, and $P_t$ is the price at the end of period $t$ (so the dividend is not included in $P_t$), the returns become
$$R_t = \frac{P_t + D_t}{P_{t-1}} - 1, \qquad r_t = \ln(P_t + D_t) - \ln(P_{t-1}).$$

**Task 9.** Write `simple_return_with_dividend(P_t_minus_1, P_t, D_t)` and `log_return_with_dividend(P_t_minus_1, P_t, D_t)`.

In [ ]:
# Task 9: write your code in place of each `raise NotImplementedError(...)` line
def simple_return_with_dividend(P_t_minus_1, P_t, D_t):
    """Simple return including the dividend D_t."""
    raise NotImplementedError("Task 9 is not completed yet")


def log_return_with_dividend(P_t_minus_1, P_t, D_t):
    """Log return including the dividend D_t."""
    raise NotImplementedError("Task 9 is not completed yet")

In [ ]:
# Tests for Task 9: run this cell. Every line should say PASS.
check(simple_return_with_dividend, 100, 102, 1, expected=0.03)
check(simple_return_with_dividend, 100, 102, 0, expected=0.02, label='no dividend: same as simple_return')
check(log_return_with_dividend, 100, 102, 1, expected=0.02955880224)

<details>
<summary><b>Show solution to Task 9</b></summary>

```python
def simple_return_with_dividend(P_t_minus_1, P_t, D_t):
    """Simple return including the dividend D_t."""
    R_t = (P_t + D_t) / P_t_minus_1 - 1
    return R_t


def log_return_with_dividend(P_t_minus_1, P_t, D_t):
    """Log return including the dividend D_t."""
    r_t = math.log(P_t + D_t) - math.log(P_t_minus_1)
    return r_t
```

</details>

### Excess return

> **Definition.** The **excess return** of an asset is its return minus the return on a reference asset, often a short-term Treasury bill:
> $$Z_t = R_t - R_{0t}, \qquad z_t = r_t - r_{0t}. \tag{1.7}$$

The excess return can be thought of as the payoff of a portfolio that is long the asset and short the reference asset, with no net initial investment.

**Task 10.** Write `excess_returns(R, R0)`, which subtracts the reference returns `R0` from the returns `R`, period by period.

In [ ]:
# Task 10: write your code in place of each `raise NotImplementedError(...)` line
def excess_returns(R, R0):
    """List of excess returns Z_t = R_t - R0_t, Eq. (1.7)."""
    raise NotImplementedError("Task 10 is not completed yet")

In [ ]:
# Tests for Task 10: run this cell. Every line should say PASS.
check(excess_returns, [0.05, 0.02], [0.01, 0.01], expected=[0.04, 0.01])
check(excess_returns, [0.0], [0.003], expected=[-0.003], label='below the Treasury-bill return')

<details>
<summary><b>Show solution to Task 10</b></summary>

```python
def excess_returns(R, R0):
    """List of excess returns Z_t = R_t - R0_t, Eq. (1.7)."""
    Z = []
    for t in range(len(R)):
        Z_t = R[t] - R0[t]
        Z.append(Z_t)
    return Z
```

</details>

**Checkpoint.** All tests of Section 1.1 should now pass.

In [ ]:
show_progress()

---
## 1.2 Distributional Properties of Returns

### 1.2.1 Moments of a random variable and their sample counterparts

The distribution of returns is summarised by its moments. For a random variable $X$ with mean $\mu_x$ and standard deviation $\sigma_x$:

* the **mean** (first moment) measures the central location, and the **variance** $\sigma_x^2$ (second central moment) measures variability;
* the **skewness** and **kurtosis** are the normalised third and fourth central moments, measuring asymmetry and tail thickness:
$$S(x) = E\Big[\frac{(X-\mu_x)^3}{\sigma_x^3}\Big], \qquad K(x) = E\Big[\frac{(X-\mu_x)^4}{\sigma_x^4}\Big].$$

For a normal distribution $S = 0$ and $K = 3$, so $K(x) - 3$ is called the **excess kurtosis**. A distribution with positive excess kurtosis has *heavy tails* (it is *leptokurtic*): extreme values are more frequent than under the normal distribution. Negative excess kurtosis means *short tails* (*platykurtic*).

In practice the moments are estimated from a sample $\{x_1, \dots, x_T\}$. We will compare our results with the book's Example 1.2, which uses the daily IBM returns. As in the book's R demonstration, `sibm` holds the percentage simple returns and `libm` the percentage log returns:

In [ ]:
ibm = da["rtn"]          # book: ibm = da[,2]
sibm = []                # book: sibm = ibm*100
libm = []                # book: libm = log(ibm+1)*100
for R_t in ibm:
    sibm.append(100 * R_t)
    libm.append(100 * math.log(1 + R_t))

print("T =", len(sibm), "   first three values of sibm:", sibm[:3])

> **Sample mean.**
> $$\hat\mu_x = \frac{1}{T}\sum_{t=1}^{T} x_t \tag{1.10}$$

**Task 11.** Write `sample_mean(x)`.

In [ ]:
# Task 11: write your code in place of each `raise NotImplementedError(...)` line
def sample_mean(x):
    """Sample mean mu_hat of the observations x = [x_1, ..., x_T], Eq. (1.10)."""
    raise NotImplementedError("Task 11 is not completed yet")

In [ ]:
# Tests for Task 11: run this cell. Every line should say PASS.
check(sample_mean, [1, 2, 3, 4], expected=2.5)
check(sample_mean, [-0.02, 0.01, 0.04], expected=0.01)
check(sample_mean, sibm, expected=0.04016125952, label='IBM daily % simple returns; book: 0.040161')

<details>
<summary><b>Show solution to Task 11</b></summary>

```python
def sample_mean(x):
    """Sample mean mu_hat of the observations x = [x_1, ..., x_T], Eq. (1.10)."""
    T = len(x)
    total = 0.0
    for x_t in x:
        total = total + x_t
    mu_hat = total / T
    return mu_hat
```

</details>

> **Sample variance and standard deviation.**
> $$\hat\sigma_x^2 = \frac{1}{T-1}\sum_{t=1}^{T}(x_t - \hat\mu_x)^2, \qquad \hat\sigma_x = \sqrt{\hat\sigma_x^2}. \tag{1.11}$$

Note the divisor $T-1$ rather than $T$: since the deviations are measured from the *estimated* mean $\hat\mu_x$, dividing by $T$ would slightly underestimate the variance. Keep this in mind for Part B, where one library divides by $T$ by default.

**Task 12.** Write `sample_variance(x)` and `sample_std(x)`.

In [ ]:
# Task 12: write your code in place of each `raise NotImplementedError(...)` line
def sample_variance(x):
    """Sample variance sigma2_hat with divisor T - 1, Eq. (1.11)."""
    raise NotImplementedError("Task 12 is not completed yet")


def sample_std(x):
    """Sample standard deviation sigma_hat, the square root of the sample variance."""
    raise NotImplementedError("Task 12 is not completed yet")

In [ ]:
# Tests for Task 12: run this cell. Every line should say PASS.
check(sample_variance, [1, 2, 3, 4], expected=1.666666667)
check(sample_variance, [2, 4, 4, 4, 5, 5, 7, 9], expected=4.571428571, label='the answer is not 4')
check(sample_std, [2, 4, 4, 4, 5, 5, 7, 9], expected=2.138089935)
check(sample_variance, sibm, expected=2.864705477, label='IBM; book: 2.864705')
check(sample_std, sibm, expected=1.692544084, label='IBM; book: 1.692544')

<details>
<summary><b>Show hint</b></summary>

First compute `mu_hat = sample_mean(x)`. Then add up the squared deviations `(x_t - mu_hat) ** 2` with an accumulator and divide by `T - 1`.

</details>

<details>
<summary><b>Show solution to Task 12</b></summary>

```python
def sample_variance(x):
    """Sample variance sigma2_hat with divisor T - 1, Eq. (1.11)."""
    T = len(x)
    mu_hat = sample_mean(x)
    total = 0.0
    for x_t in x:
        total = total + (x_t - mu_hat) ** 2
    sigma2_hat = total / (T - 1)
    return sigma2_hat


def sample_std(x):
    """Sample standard deviation sigma_hat, the square root of the sample variance."""
    sigma2_hat = sample_variance(x)
    sigma_hat = sigma2_hat ** 0.5
    return sigma_hat
```

</details>

> **Sample skewness.**
> $$\hat S(x) = \frac{1}{(T-1)\,\hat\sigma_x^3}\sum_{t=1}^{T}(x_t - \hat\mu_x)^3 \tag{1.12}$$

**Task 13.** Write `sample_skewness(x)`. Reuse `sample_mean` and `sample_std`.

In [ ]:
# Task 13: write your code in place of each `raise NotImplementedError(...)` line
def sample_skewness(x):
    """Sample skewness S_hat, Eq. (1.12)."""
    raise NotImplementedError("Task 13 is not completed yet")

In [ ]:
# Tests for Task 13: run this cell. Every line should say PASS.
check(sample_skewness, [1, 2, 3], expected=0.0, label='symmetric data')
check(sample_skewness, [1, 2, 3, 10], expected=0.8818163074, label='long right tail')
check(sample_skewness, [-10, -3, -2, -1], expected=-0.8818163074, label='mirror image: the sign flips')
check(sample_skewness, sibm, expected=0.06140501463, label="IBM; book's R output: 0.061399, see Remark 1")

<details>
<summary><b>Show hint</b></summary>

The structure is the same as `sample_variance`, with cubes `(x_t - mu_hat) ** 3`. The denominator is `(T - 1) * sigma_hat ** 3`.

</details>

<details>
<summary><b>Show solution to Task 13</b></summary>

```python
def sample_skewness(x):
    """Sample skewness S_hat, Eq. (1.12)."""
    T = len(x)
    mu_hat = sample_mean(x)
    sigma_hat = sample_std(x)
    total = 0.0
    for x_t in x:
        total = total + (x_t - mu_hat) ** 3
    S_hat = total / ((T - 1) * sigma_hat ** 3)
    return S_hat
```

</details>

> **Sample kurtosis.**
> $$\hat K(x) = \frac{1}{(T-1)\,\hat\sigma_x^4}\sum_{t=1}^{T}(x_t - \hat\mu_x)^4, \tag{1.13}$$
> and the sample excess kurtosis is $\hat K(x) - 3$.

**Task 14.** Write `sample_kurtosis(x)` and `excess_kurtosis(x)`.

In [ ]:
# Task 14: write your code in place of each `raise NotImplementedError(...)` line
def sample_kurtosis(x):
    """Sample kurtosis K_hat, Eq. (1.13). It is close to 3 for normal data."""
    raise NotImplementedError("Task 14 is not completed yet")


def excess_kurtosis(x):
    """Sample excess kurtosis K_hat - 3."""
    raise NotImplementedError("Task 14 is not completed yet")

In [ ]:
# Tests for Task 14: run this cell. Every line should say PASS.
check(sample_kurtosis, [1, 2, 3, 10], expected=1.6728)
check(sample_kurtosis, [-1, 1, -1, 1], expected=0.75, label='no tails at all')
check(excess_kurtosis, [-1, 1, -1, 1], expected=-2.25)
check(excess_kurtosis, sibm, expected=9.917671592, label="IBM; book's R output: 9.916359, see Remark 1")

<details>
<summary><b>Show solution to Task 14</b></summary>

```python
def sample_kurtosis(x):
    """Sample kurtosis K_hat, Eq. (1.13). It is close to 3 for normal data."""
    T = len(x)
    mu_hat = sample_mean(x)
    sigma_hat = sample_std(x)
    total = 0.0
    for x_t in x:
        total = total + (x_t - mu_hat) ** 4
    K_hat = total / ((T - 1) * sigma_hat ** 4)
    return K_hat


def excess_kurtosis(x):
    """Sample excess kurtosis K_hat - 3."""
    K_hat = sample_kurtosis(x)
    return K_hat - 3
```

</details>

> **Remark 1.** For the IBM returns your functions give a skewness of 0.061405 and an excess kurtosis of 9.9177, whereas the R output printed in the book shows 0.061399 and 9.9164. The R package used in the book divides by $T$ where Eqs. (1.12) and (1.13) divide by $T-1$. Both are legitimate estimators, and with almost 10,000 observations the difference is negligible. Part B shows that Python libraries use yet other conventions, so always check which definition a function implements.

### Tests of normality

Under the assumption of normality, for large $T$ the sample skewness and excess kurtosis are approximately normally distributed with mean zero and variances $6/T$ and $24/T$ (Snedecor and Cochran, 1980). This gives simple test statistics. For a significance level $\alpha$ we reject the null hypothesis if $|t| > Z_{\alpha/2}$, the upper $100(\alpha/2)$th quantile of the standard normal distribution ($Z_{0.025} = 1.96$), or equivalently if the p-value is smaller than $\alpha$.

To compute p-values we need the standard normal cumulative distribution function $\Phi$. It can be written with the error function, which is available as `math.erf`:
$$\Phi(z) = \tfrac{1}{2}\Big[1 + \operatorname{erf}\big(z/\sqrt{2}\big)\Big], \qquad \text{p-value of a two-sided test} = 2\,\big[1 - \Phi(|t|)\big].$$

**Task 15.** Write `normal_cdf(z)` and `two_sided_p_value(t)`.

In [ ]:
# Task 15: write your code in place of each `raise NotImplementedError(...)` line
def normal_cdf(z):
    """Standard normal cumulative distribution function Phi(z)."""
    raise NotImplementedError("Task 15 is not completed yet")


def two_sided_p_value(t):
    """p-value of a two-sided test whose statistic t is N(0, 1) under the null hypothesis."""
    raise NotImplementedError("Task 15 is not completed yet")

In [ ]:
# Tests for Task 15: run this cell. Every line should say PASS.
check(normal_cdf, 0, expected=0.5)
check(normal_cdf, 1.96, expected=0.9750021049)
check(normal_cdf, -1.645, expected=0.04998490554)
check(two_sided_p_value, 1.96, expected=0.0499957903, label='the familiar 5% critical value')
check(two_sided_p_value, -1.96, expected=0.0499957903, label='the sign does not matter')
check(two_sided_p_value, 0, expected=1.0)

<details>
<summary><b>Show solution to Task 15</b></summary>

```python
def normal_cdf(z):
    """Standard normal cumulative distribution function Phi(z)."""
    return 0.5 * (1 + math.erf(z / 2 ** 0.5))


def two_sided_p_value(t):
    """p-value of a two-sided test whose statistic t is N(0, 1) under the null hypothesis."""
    p = 2 * (1 - normal_cdf(abs(t)))
    return p
```

</details>

**Testing a zero mean.** Exercise 1.1(d) asks whether the mean return is zero. The usual test statistic divides the sample mean by its standard error,
$$t = \frac{\hat\mu_x}{\hat\sigma_x/\sqrt{T}} .$$
This is the statistic reported by R's `t.test` in the book. R computes the p-value from the $t$ distribution with $T-1$ degrees of freedom; for samples of hundreds or thousands of observations the normal distribution gives practically the same answer, and we use it here.

**Task 16.** Write `mean_t_test(x)`, returning the pair `t, p` (the test statistic and its p-value).

In [ ]:
# Task 16: write your code in place of each `raise NotImplementedError(...)` line
def mean_t_test(x):
    """Test of H0: mean = 0. Returns t, p."""
    raise NotImplementedError("Task 16 is not completed yet")

In [ ]:
# Tests for Task 16: run this cell. Every line should say PASS.
check(mean_t_test, [1, 2, 3, 4], expected=(3.872983346, 0.0001075111767))
check(mean_t_test, [-1, 1], expected=(0.0, 1.0), label='the mean is exactly zero')
check(mean_t_test, libm, expected=(1.51262079, 0.1303760103), label='IBM daily % log returns; book: t = 1.5126, p = 0.1304')

<details>
<summary><b>Show hint</b></summary>

A function can return two values separated by a comma: `return t, p`. The square root of `T` is `T ** 0.5`.

</details>

<details>
<summary><b>Show solution to Task 16</b></summary>

```python
def mean_t_test(x):
    """Test of H0: mean = 0. Returns t, p."""
    T = len(x)
    mu_hat = sample_mean(x)
    sigma_hat = sample_std(x)
    t = mu_hat / (sigma_hat / T ** 0.5)
    p = two_sided_p_value(t)
    return t, p
```

</details>

> **Skewness and kurtosis tests.** To test $H_0: S(r) = 0$ against $H_a: S(r) \neq 0$, and $H_0: K(r) - 3 = 0$ against $H_a: K(r) - 3 \neq 0$, use
> $$t = \frac{\hat S(r)}{\sqrt{6/T}} \qquad\text{and}\qquad t = \frac{\hat K(r) - 3}{\sqrt{24/T}} .$$

**Task 17.** Write `skewness_test(x)` and `kurtosis_test(x)`, each returning `t, p`.

In [ ]:
# Task 17: write your code in place of each `raise NotImplementedError(...)` line
def skewness_test(x):
    """Test of H0: skewness = 0. Returns t, p."""
    raise NotImplementedError("Task 17 is not completed yet")


def kurtosis_test(x):
    """Test of H0: excess kurtosis = 0. Returns t, p."""
    raise NotImplementedError("Task 17 is not completed yet")

In [ ]:
# Tests for Task 17: run this cell. Every line should say PASS.
check(skewness_test, [1, 2, 3], expected=(0.0, 1.0), label='symmetric data')
check(skewness_test, sibm, expected=(2.48734527, 0.01287004248), label='Example 1.2; book: t = 2.49, p = 0.013')
check(kurtosis_test, [-1, 1, -1, 1], expected=(-0.9185586535, 0.3583264667))
check(kurtosis_test, sibm, expected=(200.8685583, 0.0), label='IBM daily % simple returns')

<details>
<summary><b>Show solution to Task 17</b></summary>

```python
def skewness_test(x):
    """Test of H0: skewness = 0. Returns t, p."""
    T = len(x)
    S_hat = sample_skewness(x)
    t = S_hat / (6 / T) ** 0.5
    p = two_sided_p_value(t)
    return t, p


def kurtosis_test(x):
    """Test of H0: excess kurtosis = 0. Returns t, p."""
    T = len(x)
    K_hat = sample_kurtosis(x)
    t = (K_hat - 3) / (24 / T) ** 0.5
    p = two_sided_p_value(t)
    return t, p
```

</details>

> **Jarque–Bera test.** Jarque and Bera (1987) combine the two tests into one statistic,
> $$JB = \frac{\hat S^2(r)}{6/T} + \frac{[\hat K(r) - 3]^2}{24/T},$$
> which is asymptotically chi-squared with 2 degrees of freedom under normality. Normality is rejected if the p-value is smaller than the significance level.

For 2 degrees of freedom the chi-squared distribution has a simple tail probability, $P(\chi^2_2 > x) = e^{-x/2}$, so the p-value is `math.exp(-JB / 2)`.

**Task 18.** Write `jarque_bera_test(x)`, returning `JB, p`.

In [ ]:
# Task 18: write your code in place of each `raise NotImplementedError(...)` line
def jarque_bera_test(x):
    """Jarque-Bera statistic and its p-value from the chi-squared(2) distribution. Returns JB, p."""
    raise NotImplementedError("Task 18 is not completed yet")

In [ ]:
# Tests for Task 18: run this cell. Every line should say PASS.
check(jarque_bera_test, [-1, 1, -1, 1], expected=(0.84375, 0.6558160113))
check(jarque_bera_test, [1, 2, 3, 10], expected=(0.81197664, 0.66631795))
check(jarque_bera_test, libm, expected=(60906.52756, 0.0), label="IBM daily % log returns; book's R output: 60921.93, see Remark 1")

<details>
<summary><b>Show hint</b></summary>

Note that $JB$ is the sum of the squares of the two $t$ statistics of Task 17.

</details>

<details>
<summary><b>Show solution to Task 18</b></summary>

```python
def jarque_bera_test(x):
    """Jarque-Bera statistic and its p-value from the chi-squared(2) distribution. Returns JB, p."""
    T = len(x)
    S_hat = sample_skewness(x)
    K_hat = sample_kurtosis(x)
    JB = S_hat ** 2 / (6 / T) + (K_hat - 3) ** 2 / (24 / T)
    p = math.exp(-JB / 2)
    return JB, p
```

</details>

**Example 1.2 of the book.** For the daily simple returns of IBM, the book reports a skewness test statistic of 2.49 with a p-value of about 0.013, concluding that the returns are significantly skewed to the right at the 5% level. For the log returns, the test of a zero mean gives $t = 1.5126$ ($p = 0.1304$), and the Jarque–Bera test rejects normality. Your functions reproduce these results:

In [ ]:
t, p = skewness_test(sibm)
print("Skewness test, simple returns:   t =", round(t, 2), "  p =", round(p, 4))

t, p = mean_t_test(libm)
print("Zero-mean test, log returns:     t =", round(t, 4), "  p =", round(p, 4))

JB, p = jarque_bera_test(libm)
print("Jarque-Bera test, log returns:   JB =", round(JB, 1), "  p =", p)

**Question 4.** The test of a zero mean for IBM's daily log returns gives $p \approx 0.13$, although the sample mean is positive. How should this result be interpreted?

<details>
<summary><b>Show answer</b></summary>

At the 5% (or 10%) level we cannot reject $H_0: \mu = 0$. Daily expected returns are very small compared with daily volatility (about 0.03% against 1.7%), so even 9,845 observations estimate the mean imprecisely. Failing to reject is not the same as proving that the mean is zero.

</details>

In [ ]:
show_progress()

---
### 1.2.2 Distributions of returns

Several distributions have been proposed for asset returns:

* **Normal distribution.** Convenient, but it has three problems: a simple return cannot be below −1 while a normal variable can take any value; a product of normal one-period returns is not normal; and observed returns tend to have positive excess kurtosis.
* **Lognormal distribution.** Assume the *log* returns are independent and identically distributed (iid) normal. Simple returns then satisfy the lower bound, and multiperiod log returns remain normal. Excess kurtosis is still not captured.
* **Stable distributions**, such as the Cauchy distribution, have heavy tails but infinite variance, which conflicts with most finance theory.
* **Scale mixtures of normal distributions.** Most of the time returns come from a normal distribution with small variance, occasionally from one with large variance. A finite mixture is $r_t \sim (1-X)\,N(\mu, \sigma_1^2) + X\,N(\mu, \sigma_2^2)$, where $X$ is Bernoulli with $P(X = 1) = \alpha$.

The probability density functions needed are
$$f_N(x) = \frac{1}{\sqrt{2\pi}\,\sigma}\exp\Big(-\frac{(x-\mu)^2}{2\sigma^2}\Big), \qquad f_C(x) = \frac{1}{\pi(1+x^2)}, \qquad f_{mix}(x) = (1-\alpha)\,f_N(x;\mu,\sigma_1) + \alpha\,f_N(x;\mu,\sigma_2).$$

Figure 1.1 of the book compares the standard normal density, the Cauchy density and the mixture with $\mu = 0$, $\sigma_1 = 1$, $\sigma_2 = 4$ and $\alpha = 0.05$.

**Task 19.** Write `normal_pdf(x, mu, sigma)`, `cauchy_pdf(x)` and `mixture_pdf(x, mu, sigma1, sigma2, alpha)`.

In [ ]:
# Task 19: write your code in place of each `raise NotImplementedError(...)` line
def normal_pdf(x, mu, sigma):
    """Normal density with mean mu and standard deviation sigma."""
    raise NotImplementedError("Task 19 is not completed yet")


def cauchy_pdf(x):
    """Standard Cauchy density."""
    raise NotImplementedError("Task 19 is not completed yet")


def mixture_pdf(x, mu, sigma1, sigma2, alpha):
    """Density of the mixture (1 - X) N(mu, sigma1^2) + X N(mu, sigma2^2), with P(X = 1) = alpha."""
    raise NotImplementedError("Task 19 is not completed yet")

In [ ]:
# Tests for Task 19: run this cell. Every line should say PASS.
check(normal_pdf, 0, 0, 1, expected=0.3989422804, label='peak of the standard normal')
check(normal_pdf, 1, 1, 2, expected=0.1994711402)
check(normal_pdf, -1.5, 0, 1, expected=0.1295175957)
check(cauchy_pdf, 0, expected=0.3183098862)
check(cauchy_pdf, 3, expected=0.03183098862)
check(mixture_pdf, 0, 0, 1, 4, 0.05, expected=0.3839819449, label='the mixture of Figure 1.1')
check(mixture_pdf, 0.7, 0, 1, 4, 0.0, expected=0.3122539334, label='alpha = 0 gives the normal density')

<details>
<summary><b>Show solution to Task 19</b></summary>

```python
def normal_pdf(x, mu, sigma):
    """Normal density with mean mu and standard deviation sigma."""
    z = (x - mu) / sigma
    return math.exp(-0.5 * z * z) / (sigma * (2 * math.pi) ** 0.5)


def cauchy_pdf(x):
    """Standard Cauchy density."""
    return 1 / (math.pi * (1 + x ** 2))


def mixture_pdf(x, mu, sigma1, sigma2, alpha):
    """Density of the mixture (1 - X) N(mu, sigma1^2) + X N(mu, sigma2^2), with P(X = 1) = alpha."""
    f1 = normal_pdf(x, mu, sigma1)
    f2 = normal_pdf(x, mu, sigma2)
    return (1 - alpha) * f1 + alpha * f2
```

</details>

The next cell compares the three densities at the centre and in the tails (Figure 1.1 itself is drawn in Part B).

In [ ]:
print("  x     normal    mixture     Cauchy   mixture/normal")
for x in [0, 1, 2, 3, 4, 5]:
    f_N = normal_pdf(x, 0, 1)
    f_mix = mixture_pdf(x, 0, 1, 4, 0.05)
    f_C = cauchy_pdf(x)
    print(f"{x:>3} {f_N:10.6f} {f_mix:10.6f} {f_C:10.6f} {f_mix / f_N:12.1f}")

**Question 5.** What does the last column tell you about extreme returns under the mixture, compared with the normal distribution?

<details>
<summary><b>Show answer</b></summary>

Near the centre the two densities are almost equal (the mixture is even slightly lower), but in the tails the mixture density is far larger: about twice the normal density at $x = 3$, about 24 times at $x = 4$ and more than 1,500 times at $x = 5$. A model that looks almost normal on ordinary days can still make large moves much more likely, which is exactly what positive excess kurtosis means. The Cauchy tails are heavier still.

</details>

> **Lognormal distribution.** If the log returns $r_t$ are iid $N(\mu, \sigma^2)$, the simple returns are lognormal with mean and variance
> $$E(R_t) = \exp\Big(\mu + \frac{\sigma^2}{2}\Big) - 1, \qquad \operatorname{Var}(R_t) = \exp(2\mu + \sigma^2)\big[\exp(\sigma^2) - 1\big]. \tag{1.17}$$
> Conversely, if the simple return has mean $m_1$ and variance $m_2$, the log return has
> $$E(r_t) = \ln\left[\frac{m_1 + 1}{\sqrt{1 + m_2/(1+m_1)^2}}\right], \qquad \operatorname{Var}(r_t) = \ln\left[1 + \frac{m_2}{(1+m_1)^2}\right].$$

**Task 20.** Write `lognormal_moments(mu, sigma2)` (Eq. 1.17) and `log_return_moments(m1, m2)`. Each returns two numbers: the mean and the variance.

In [ ]:
# Task 20: write your code in place of each `raise NotImplementedError(...)` line
def lognormal_moments(mu, sigma2):
    """Mean and variance of the simple return R_t when r_t is N(mu, sigma2), Eq. (1.17)."""
    raise NotImplementedError("Task 20 is not completed yet")


def log_return_moments(m1, m2):
    """Mean and variance of r_t when the simple return has mean m1 and variance m2."""
    raise NotImplementedError("Task 20 is not completed yet")

In [ ]:
# Tests for Task 20: run this cell. Every line should say PASS.
check(lognormal_moments, 0.0, 0.0, expected=(0.0, 0.0), label='no risk, no drift')
check(lognormal_moments, 0.01, 0.0025, expected=(0.01131351922, 0.00256008636))
check(lognormal_moments, 0.0, 0.04, expected=(0.02020134003, 0.04247629348))
check(log_return_moments, 0.01131351922, 0.00256008636, expected=(0.009999999996, 0.0025), label='inverse of lognormal_moments(0.01, 0.0025)')

<details>
<summary><b>Show solution to Task 20</b></summary>

```python
def lognormal_moments(mu, sigma2):
    """Mean and variance of the simple return R_t when r_t is N(mu, sigma2), Eq. (1.17)."""
    mean_R = math.exp(mu + sigma2 / 2) - 1
    var_R = math.exp(2 * mu + sigma2) * (math.exp(sigma2) - 1)
    return mean_R, var_R


def log_return_moments(m1, m2):
    """Mean and variance of r_t when the simple return has mean m1 and variance m2."""
    ratio = m2 / (1 + m1) ** 2
    mean_r = math.log((m1 + 1) / (1 + ratio) ** 0.5)
    var_r = math.log(1 + ratio)
    return mean_r, var_r
```

</details>

**Question 6.** With $\mu = 0$ and $\sigma^2 = 0.04$ the expected **log** return is zero, yet the expected **simple** return is about +2%. Where does the extra 2% come from?

<details>
<summary><b>Show answer</b></summary>

From the convexity of the exponential function (Jensen's inequality): $E[e^{r}] > e^{E[r]}$. Gains add more in simple-return terms than equally large losses take away, so $E(R_t) = e^{\sigma^2/2} - 1 \approx \sigma^2/2 = 2\%$. It is the mirror image of the gap between arithmetic and geometric averages in Question 2.

</details>

---
### 1.2.3 Multivariate returns

For $N$ assets, the returns at time $t$ form a vector $\mathbf{x}_t = (x_{1t}, \dots, x_{Nt})'$. Given data $\{\mathbf{x}_1, \dots, \mathbf{x}_T\}$, the sample mean vector and covariance matrix are
$$\hat{\boldsymbol\mu}_x = \frac{1}{T}\sum_{t=1}^{T}\mathbf{x}_t, \qquad \hat{\boldsymbol\Sigma}_x = \frac{1}{T-1}\sum_{t=1}^{T}(\mathbf{x}_t - \hat{\boldsymbol\mu}_x)(\mathbf{x}_t - \hat{\boldsymbol\mu}_x)'.$$

Element $(i, j)$ of $\hat{\boldsymbol\Sigma}_x$ is $\frac{1}{T-1}\sum_{t}(x_{it} - \hat\mu_i)(x_{jt} - \hat\mu_j)$, the sample covariance of assets $i$ and $j$; the diagonal elements are the sample variances.

In plain Python we store the data as a **list of rows**: `X[t]` is the list of the $N$ returns at time $t$, so `X[t][i]` is $x_{it}$. A matrix is likewise a list of rows. The next cell builds `X` for the three stocks of Exercise 1.1:

In [ ]:
d3 = load_data("d-3stocks9908.txt")
X = []
for t in range(len(d3["axp"])):
    X.append([100 * d3["axp"][t], 100 * d3["cat"][t], 100 * d3["sbux"][t]])   # AXP, CAT, SBUX in %

print("T =", len(X), "rows;  first row X[0] =", X[0])

**Task 21.** Write `sample_mean_vector(X)` and `sample_covariance_matrix(X)`.

In [ ]:
# Task 21: write your code in place of each `raise NotImplementedError(...)` line
def sample_mean_vector(X):
    """X is a list of T rows with N numbers each. Returns the list of the N column means."""
    raise NotImplementedError("Task 21 is not completed yet")


def sample_covariance_matrix(X):
    """N x N sample covariance matrix Sigma_hat (divisor T - 1), as a list of N rows."""
    raise NotImplementedError("Task 21 is not completed yet")

In [ ]:
# Tests for Task 21: run this cell. Every line should say PASS.
check(sample_mean_vector, [[1, 2], [3, 4], [5, 9]], expected=[3.0, 5.0])
check(sample_covariance_matrix, [[1, 2], [3, 4], [5, 9]], expected=[[4.0, 7.0], [7.0, 13.0]])
check(sample_mean_vector, X, expected=[0.01456536779, 0.05950365805, 0.04805367793], label='AXP, CAT, SBUX daily % returns')
check(sample_covariance_matrix, X, expected=[[5.983984484, 2.464209551, 2.507915172], [2.464209551, 4.707373336, 1.72045575], [2.507915172, 1.72045575, 7.196460098]], label='AXP, CAT, SBUX daily % returns')

<details>
<summary><b>Show hint</b></summary>

For the mean vector, loop over the assets `i`; for each asset add up `X[t][i]` over `t` and divide by `T`.
For the covariance matrix use three nested loops: over `i`, over `j`, and inside them over `t`, adding `(X[t][i] - mu_hat[i]) * (X[t][j] - mu_hat[j])`. Collect each finished row in a list and append it to `Sigma_hat`.

</details>

<details>
<summary><b>Show solution to Task 21</b></summary>

```python
def sample_mean_vector(X):
    """X is a list of T rows with N numbers each. Returns the list of the N column means."""
    T = len(X)
    N = len(X[0])
    mu_hat = []
    for i in range(N):
        total = 0.0
        for t in range(T):
            total = total + X[t][i]
        mu_hat.append(total / T)
    return mu_hat


def sample_covariance_matrix(X):
    """N x N sample covariance matrix Sigma_hat (divisor T - 1), as a list of N rows."""
    T = len(X)
    N = len(X[0])
    mu_hat = sample_mean_vector(X)
    Sigma_hat = []
    for i in range(N):
        row = []
        for j in range(N):
            total = 0.0
            for t in range(T):
                total = total + (X[t][i] - mu_hat[i]) * (X[t][j] - mu_hat[j])
            row.append(total / (T - 1))
        Sigma_hat.append(row)
    return Sigma_hat
```

</details>

---
### 1.2.4 The likelihood function of returns

Writing the joint density of $r_1, \dots, r_T$ as a product of conditional densities gives the likelihood function in Eq. (1.18) of the book. If each conditional distribution is normal with mean $\mu_t$ and variance $\sigma_t^2$, the log-likelihood is a sum of terms $-\tfrac{1}{2}\big[\ln(2\pi) + \ln\sigma_t^2 + (r_t - \mu_t)^2/\sigma_t^2\big]$.

In the simplest special case the returns are iid $N(\mu, \sigma^2)$: every term has the same $\mu$ and $\sigma^2$, the first observation is treated like the others, and
$$\ln L(\mu, \sigma^2) = -\frac{1}{2}\sum_{t=1}^{T}\left[\ln(2\pi) + \ln\sigma^2 + \frac{(r_t - \mu)^2}{\sigma^2}\right].$$
The values of $\mu$ and $\sigma^2$ that maximise $\ln L$ are the **maximum-likelihood estimates (MLE)**.

**Task 22.** Write `normal_log_likelihood(r, mu, sigma2)` for a list of returns `r`.

In [ ]:
# Task 22: write your code in place of each `raise NotImplementedError(...)` line
def normal_log_likelihood(r, mu, sigma2):
    """Log-likelihood of the returns r under the iid N(mu, sigma2) model."""
    raise NotImplementedError("Task 22 is not completed yet")

In [ ]:
# Tests for Task 22: run this cell. Every line should say PASS.
check(normal_log_likelihood, [0.0], 0.0, 1.0, expected=-0.9189385332, label='one observation at the mean')
check(normal_log_likelihood, [1, -1], 0.0, 1.0, expected=-2.837877066)
check(normal_log_likelihood, [0.01, -0.02, 0.03], 0.0, 0.0004, expected=7.229253417)

<details>
<summary><b>Show solution to Task 22</b></summary>

```python
def normal_log_likelihood(r, mu, sigma2):
    """Log-likelihood of the returns r under the iid N(mu, sigma2) model."""
    total = 0.0
    for r_t in r:
        total = total + math.log(2 * math.pi) + math.log(sigma2) + (r_t - mu) ** 2 / sigma2
    return -0.5 * total
```

</details>

**Maximum likelihood by trial and error.** For the monthly log returns of the S&P composite index, the next cell tries 401 values of $\mu$ between −2% and +2% (holding $\sigma^2$ at the sample variance) and keeps the value with the highest log-likelihood.

In [ ]:
dm = load_data("m-gm3dx7508.txt")
r_sp = []
for R_t in dm["sp"]:
    r_sp.append(math.log(1 + R_t))
sigma2_hat = sample_variance(r_sp)

best_mu = 0.0
best_lnL = normal_log_likelihood(r_sp, best_mu, sigma2_hat)
for i in range(-200, 201):
    mu = i / 10000
    lnL = normal_log_likelihood(r_sp, mu, sigma2_hat)
    if lnL > best_lnL:
        best_mu = mu
        best_lnL = lnL

print("MLE of mu (grid search):", best_mu)
print("sample mean:            ", round(sample_mean(r_sp), 6))

**Question 7.** The grid search finds (the grid point closest to) the sample mean. Which value of $\sigma^2$ maximises the likelihood, and how does it differ from Eq. (1.11)?

<details>
<summary><b>Show answer</b></summary>

Setting the derivative with respect to $\sigma^2$ to zero gives $\hat\sigma^2_{ML} = \frac{1}{T}\sum_t (r_t - \hat\mu)^2$: the divisor is $T$, not $T-1$. The maximum-likelihood estimator of the variance is slightly biased downwards, which Eq. (1.11) corrects.

</details>

In [ ]:
show_progress()

---
### 1.2.5 Empirical properties of returns

Table 1.2 of the book reports descriptive statistics for daily and monthly returns of several indexes and stocks. The daily rows for the S&P composite (SP), the value-weighted (VW) and equal-weighted (EW) CRSP indexes and IBM can be reproduced from `d-ibm3dx7008.txt`. Compare the output with the book; small differences in the last digit of the kurtosis come from the divisor discussed in Remark 1.

In [ ]:
columns = ["sprtrn", "vwretd", "ewretd", "rtn"]
labels = ["SP", "VW", "EW", "IBM"]

for kind in ["simple", "log"]:
    print(f"Daily {kind} returns (%)")
    print("          Mean   St.dev.  Skewness  Ex.kurt.   Minimum  Maximum")
    for i in range(len(columns)):
        x = []
        for R_t in da[columns[i]]:
            if kind == "simple":
                x.append(100 * R_t)
            else:
                x.append(100 * math.log(1 + R_t))
        print(f"{labels[i]:<6}{sample_mean(x):8.3f}{sample_std(x):10.3f}{sample_skewness(x):10.2f}"
              f"{excess_kurtosis(x):10.2f}{min(x):10.2f}{max(x):9.2f}")
    print()

From Table 1.2 the book draws several empirical regularities: (a) daily returns of market indexes and individual stocks tend to have high excess kurtosis; (b) the mean of a daily return series is close to zero; (c) monthly returns have higher standard deviations than daily returns; (d) among daily returns, market indexes have smaller standard deviations than individual stocks; (e) skewness is not a serious problem; and (f) the difference between simple and log returns is small. The exercises below let you check these regularities on other data.

---
## 1.3 Exercises of Chapter 1

The data sets of these exercises come from the book's website and cover the periods stated in the book: daily stock returns 1999 to 2008, monthly returns 1975 to 2008, and daily exchange rates from January 2000 to March 2009.

First, three small helper functions that the exercises use repeatedly.

**Task 23.** Write `to_percent(x)`, which multiplies every element of a list by 100, and `simple_to_log_list(R)`, which converts a list of simple returns into a list of log returns (use your `simple_to_log`).

In [ ]:
# Task 23: write your code in place of each `raise NotImplementedError(...)` line
def to_percent(x):
    """List with every element of x multiplied by 100."""
    raise NotImplementedError("Task 23 is not completed yet")


def simple_to_log_list(R):
    """List of log returns r_t = ln(1 + R_t) from a list of simple returns R."""
    raise NotImplementedError("Task 23 is not completed yet")

In [ ]:
# Tests for Task 23: run this cell. Every line should say PASS.
check(to_percent, [0.01, -0.025, 0.1], expected=[1.0, -2.5, 10.0])
check(simple_to_log_list, [0.05, -0.5], expected=[0.04879016417, -0.6931471806])

<details>
<summary><b>Show solution to Task 23</b></summary>

```python
def to_percent(x):
    """List with every element of x multiplied by 100."""
    x_pct = []
    for x_t in x:
        x_pct.append(100 * x_t)
    return x_pct


def simple_to_log_list(R):
    """List of log returns r_t = ln(1 + R_t) from a list of simple returns R."""
    r = []
    for R_t in R:
        r.append(simple_to_log(R_t))
    return r
```

</details>

**Task 24.** Write `summary_statistics(x)`, returning a dictionary with the keys `"n"`, `"mean"`, `"std"`, `"skew"`, `"ex_kurt"`, `"min"` and `"max"`. A dictionary is written as `{"n": T, "mean": mu_hat, ...}`.

In [ ]:
# Task 24: write your code in place of each `raise NotImplementedError(...)` line
def summary_statistics(x):
    """Dictionary with the sample size, mean, standard deviation, skewness, excess kurtosis, minimum and maximum."""
    raise NotImplementedError("Task 24 is not completed yet")

In [ ]:
# Tests for Task 24: run this cell. Every line should say PASS.
check(summary_statistics, [1, 2, 3, 10], expected={'n': 4, 'mean': 4.0, 'std': 4.082482905, 'skew': 0.8818163074, 'ex_kurt': -1.3272, 'min': 1, 'max': 10})

<details>
<summary><b>Show solution to Task 24</b></summary>

```python
def summary_statistics(x):
    """Dictionary with the sample size, mean, standard deviation, skewness, excess kurtosis, minimum and maximum."""
    stats = {
        "n": len(x),
        "mean": sample_mean(x),
        "std": sample_std(x),
        "skew": sample_skewness(x),
        "ex_kurt": excess_kurtosis(x),
        "min": min(x),
        "max": max(x),
    }
    return stats
```

</details>

The next cell provides two printing functions that lay out your results as a table, with the same columns as the course solutions: the summary statistics plus the $t$ statistic of the zero-mean test and its p-value. Run it; you do not need to change it.

In [ ]:
def print_header():
    print("            n      Mean   St.dev.  Skewness  Ex.kurt.    Minimum   Maximum   t-ratio  p-value")


def print_row(name, x):
    if x is None:
        print(f"{name:<6}  (not computed yet)")
        return
    s = summary_statistics(x)
    t, p = mean_t_test(x)
    print(f"{name:<6}{s['n']:>7}{s['mean']:10.4f}{s['std']:10.4f}{s['skew']:10.4f}{s['ex_kurt']:10.4f}"
          f"{s['min']:11.4f}{s['max']:10.4f}{t:10.3f}{p:9.4f}")

### Exercise 1.1

Consider the daily stock returns of American Express (AXP), Caterpillar (CAT) and Starbucks (SBUX) from January 1999 to December 2008. The data are simple returns in the file `d-3stocks9908.txt` (date, axp, cat, sbux).

(a) Express the simple returns in percentages. Compute the sample mean, standard deviation, skewness, excess kurtosis, minimum and maximum of the percentage simple returns.

(b) Transform the simple returns to log returns.

(c) Express the log returns in percentages. Compute the same statistics for the percentage log returns.

(d) Test the null hypothesis that the mean of the log returns of each stock is zero, using the 5% significance level.

**Task 25.** Solve Exercise 1.1. Replace each `None` with the requested list; the cell after the tests prints the tables for parts (a), (c) and (d).

In [ ]:
# Task 25: write your code in place of each `raise NotImplementedError(...)` line
d3 = load_data("d-3stocks9908.txt")
R_axp = d3["axp"]          # daily simple returns, in decimals
R_cat = d3["cat"]
R_sbux = d3["sbux"]

# (a) percentage simple returns
R_axp_pct = None
R_cat_pct = None
R_sbux_pct = None

# (b) and (c) percentage log returns
r_axp_pct = None
r_cat_pct = None
r_sbux_pct = None

In [ ]:
# Tests for Task 25: run this cell. Every line should say PASS.
check(summary_statistics, R_axp_pct, expected={'n': 2515, 'mean': 0.01456536779, 'std': 2.446218405, 'skew': -0.03461989198, 'ex_kurt': 6.051650379, 'min': -17.5949, 'max': 17.9266}, label='Ex. 1.1(a), AXP')
check(summary_statistics, r_sbux_pct, expected={'n': 2515, 'mean': 0.01188537469, 'std': 2.695887765, 'skew': -0.5973054713, 'ex_kurt': 12.90179611, 'min': -33.24869883, 'max': 13.65864711}, label='Ex. 1.1(c), SBUX')
check(mean_t_test, r_cat_pct, expected=(0.8302265106, 0.4064107291), label='Ex. 1.1(d), CAT')

In [ ]:
print("(a) Percentage simple returns")
print_header()
print_row("AXP", R_axp_pct)
print_row("CAT", R_cat_pct)
print_row("SBUX", R_sbux_pct)

print()
print("(c) and (d) Percentage log returns")
print_header()
print_row("AXP", r_axp_pct)
print_row("CAT", r_cat_pct)
print_row("SBUX", r_sbux_pct)

<details>
<summary><b>Show hint</b></summary>

Your helper functions do all the work: `to_percent(R_axp)` for (a), and `to_percent(simple_to_log_list(R_axp))` for (b) and (c).

</details>

<details>
<summary><b>Show solution to Task 25</b></summary>

```python
d3 = load_data("d-3stocks9908.txt")
R_axp = d3["axp"]          # daily simple returns, in decimals
R_cat = d3["cat"]
R_sbux = d3["sbux"]

# (a) percentage simple returns
R_axp_pct = to_percent(R_axp)
R_cat_pct = to_percent(R_cat)
R_sbux_pct = to_percent(R_sbux)

# (b) and (c) percentage log returns
r_axp_pct = to_percent(simple_to_log_list(R_axp))
r_cat_pct = to_percent(simple_to_log_list(R_cat))
r_sbux_pct = to_percent(simple_to_log_list(R_sbux))
```

With this solution, the cell that prints the results shows:

```text
(a) Percentage simple returns
            n      Mean   St.dev.  Skewness  Ex.kurt.    Minimum   Maximum   t-ratio  p-value
AXP      2515    0.0146    2.4462   -0.0346    6.0517   -17.5949   17.9266     0.299   0.7652
CAT      2515    0.0595    2.1696    0.0117    4.4562   -14.5175   14.7229     1.375   0.1690
SBUX     2515    0.0481    2.6826   -0.0825    8.7503   -28.2862   14.6354     0.898   0.3690

(c) and (d) Percentage log returns
            n      Mean   St.dev.  Skewness  Ex.kurt.    Minimum   Maximum   t-ratio  p-value
AXP      2515   -0.0154    2.4529   -0.3366    6.4903   -19.3523   16.4892    -0.316   0.7523
CAT      2515    0.0359    2.1715   -0.2018    4.6978   -15.6859   13.7349     0.830   0.4064
SBUX     2515    0.0119    2.6959   -0.5973   12.9018   -33.2487   13.6586     0.221   0.8250
```

**(d) Conclusion.** All three p-values are well above 0.05, so for none of the stocks can we reject the hypothesis that the mean daily log return is zero.

</details>

**Question 8.** None of the three mean returns is significantly different from zero, yet each stock clearly gained or lost value over the decade. Is there a contradiction?

<details>
<summary><b>Show answer</b></summary>

No. Ten years of daily data estimate the *volatility* very precisely but the *mean* only roughly: the standard error of a daily mean is about $2.5\% / \sqrt{2515} \approx 0.05\%$, larger than the means themselves. Notice also that the AXP mean is positive for simple returns but negative for log returns; the log mean is lower by roughly half the variance (compare Questions 2 and 6).

</details>

### Exercise 1.2

Answer the same questions as in Exercise 1.1, but using the monthly returns of General Motors (GM), the CRSP value-weighted index (VW), the CRSP equal-weighted index (EW) and the S&P composite index (SP) from January 1975 to December 2008. The index returns include dividend distributions. The data file is `m-gm3dx7508.txt` (date, gm, vw, ew, sp).

**Task 26.** Solve Exercise 1.2 in the same way as Exercise 1.1.

In [ ]:
# Task 26: write your code in place of each `raise NotImplementedError(...)` line
dm = load_data("m-gm3dx7508.txt")
R_gm = dm["gm"]            # monthly simple returns, in decimals
R_vw = dm["vw"]
R_ew = dm["ew"]
R_sp = dm["sp"]

# (a) percentage simple returns
R_gm_pct = None
R_vw_pct = None
R_ew_pct = None
R_sp_pct = None

# (b) and (c) percentage log returns
r_gm_pct = None
r_vw_pct = None
r_ew_pct = None
r_sp_pct = None

In [ ]:
# Tests for Task 26: run this cell. Every line should say PASS.
check(summary_statistics, R_gm_pct, expected={'n': 408, 'mean': 0.556754902, 'std': 9.272703947, 'skew': -0.3844170435, 'ex_kurt': 2.060479249, 'min': -38.9313, 'max': 27.6619}, label='Ex. 1.2(a), GM')
check(summary_statistics, r_vw_pct, expected={'n': 408, 'mean': 0.9045670044, 'std': 4.560585388, 'skew': -1.053583024, 'ex_kurt': 3.954593437, 'min': -25.53607465, 'max': 13.24307872}, label='Ex. 1.2(c), VW')
check(mean_t_test, r_sp_pct, expected=(2.899570196, 0.003736746632), label='Ex. 1.2(d), SP')
check(mean_t_test, r_gm_pct, expected=(0.2320649947, 0.8164875329), label='Ex. 1.2(d), GM')

In [ ]:
print("(a) Percentage simple returns")
print_header()
print_row("GM", R_gm_pct)
print_row("VW", R_vw_pct)
print_row("EW", R_ew_pct)
print_row("SP", R_sp_pct)

print()
print("(c) and (d) Percentage log returns")
print_header()
print_row("GM", r_gm_pct)
print_row("VW", r_vw_pct)
print_row("EW", r_ew_pct)
print_row("SP", r_sp_pct)

<details>
<summary><b>Show solution to Task 26</b></summary>

```python
dm = load_data("m-gm3dx7508.txt")
R_gm = dm["gm"]            # monthly simple returns, in decimals
R_vw = dm["vw"]
R_ew = dm["ew"]
R_sp = dm["sp"]

# (a) percentage simple returns
R_gm_pct = to_percent(R_gm)
R_vw_pct = to_percent(R_vw)
R_ew_pct = to_percent(R_ew)
R_sp_pct = to_percent(R_sp)

# (b) and (c) percentage log returns
r_gm_pct = to_percent(simple_to_log_list(R_gm))
r_vw_pct = to_percent(simple_to_log_list(R_vw))
r_ew_pct = to_percent(simple_to_log_list(R_ew))
r_sp_pct = to_percent(simple_to_log_list(R_sp))
```

With this solution, the cell that prints the results shows:

```text
(a) Percentage simple returns
            n      Mean   St.dev.  Skewness  Ex.kurt.    Minimum   Maximum   t-ratio  p-value
GM        408    0.5568    9.2727   -0.3844    2.0605   -38.9313   27.6619     1.213   0.2252
VW        408    1.0118    4.5075   -0.7445    2.6800   -22.5363   14.1600     4.534   0.0000
EW        408    1.3314    5.5959   -0.3009    4.3517   -27.2248   29.9260     4.806   0.0000
SP        408    0.7301    4.3599   -0.5719    2.2815   -21.7630   13.1767     3.382   0.0007

(c) and (d) Percentage log returns
            n      Mean   St.dev.  Skewness  Ex.kurt.    Minimum   Maximum   t-ratio  p-value
GM        408    0.1102    9.5903   -1.0262    4.0380   -49.3171   24.4215     0.232   0.8165
VW        408    0.9046    4.5606   -1.0536    3.9546   -25.5361   13.2431     4.006   0.0001
EW        408    1.1670    5.6260   -0.8382    5.2627   -31.7795   26.1795     4.190   0.0000
SP        408    0.6319    4.4022   -0.8569    3.3503   -24.5428   12.3780     2.900   0.0037
```

**(d) Conclusion.** For the three indexes the p-values are below 0.05, so the mean monthly log return is significantly positive at the 5% level. For GM the p-value is about 0.82, so its mean log return is not significantly different from zero.

</details>

**Question 9.** Why can the mean of the monthly **index** returns be distinguished from zero when the daily stock means of Exercise 1.1 could not, and why does GM still fail?

<details>
<summary><b>Show answer</b></summary>

A monthly mean is about 21 times a daily mean, while the monthly standard deviation is only about $\sqrt{21} \approx 4.6$ times the daily one, so each observation carries more information about the mean. Diversified indexes are also less volatile (about 4.5% per month) than a single stock. GM combines a high volatility (9.6% per month) with a mean log return close to zero over 1975 to 2008, so its t statistic stays small.

</details>

### Exercise 1.3

Consider the monthly stock returns of the S&P composite index from January 1975 to December 2008 in Exercise 1.2.

(a) What is the average annual log return over the data span?

(b) Assume that there were no transaction costs. If one invested \$1.00 in the S&P composite index at the beginning of 1975, what was the value of the investment at the end of 2008?

**Task 27.** Solve Exercise 1.3. Store the monthly log returns in `r_sp`, the number of years in `n`, the average annual log return (in decimals) in `r_annual`, and the final value of the investment in `V`.

In [ ]:
# Task 27: write your code in place of each `raise NotImplementedError(...)` line
R_sp = load_data("m-gm3dx7508.txt")["sp"]     # monthly simple returns of the S&P index, in decimals

r_sp = None         # monthly log returns, in decimals
n = None            # number of years in the data span
r_annual = None     # (a) average annual log return
V = None            # (b) value of $1.00 at the end of 2008

In [ ]:
# Tests for Task 27: run this cell. Every line should say PASS.
check_value('Ex. 1.3: number of years', n, expected=34.0)
check_value('Ex. 1.3(a): average annual log return', r_annual, expected=0.07583246929)
check_value('Ex. 1.3(b): value of $1.00', V, expected=13.1747742)

In [ ]:
if V is None:
    print("Complete the task above first.")
else:
    print("number of years:              ", n)
    print("(a) average annual log return:", round(100 * r_annual, 3), "%")
    print("(b) value of $1.00 at the end: $", round(V, 2))
    print("    check with simple returns: $", round(multiperiod_gross_return(R_sp), 2))

<details>
<summary><b>Show hint</b></summary>

Log returns add up over time, so the total log return is `multiperiod_log_return(r_sp)`, and the average annual log return is the total divided by the number of years (equivalently, 12 times the monthly mean). By Eq. (1.4), \$1.00 grows to $\exp(r \times n)$.

</details>

<details>
<summary><b>Show solution to Task 27</b></summary>

```python
R_sp = load_data("m-gm3dx7508.txt")["sp"]     # monthly simple returns of the S&P index, in decimals

r_sp = simple_to_log_list(R_sp)                      # monthly log returns, in decimals
n = len(r_sp) / 12                                   # number of years in the data span
r_annual = multiperiod_log_return(r_sp) / n          # (a) average annual log return
V = continuous_compound_value(1.0, r_annual, n)      # (b) value of $1.00 at the end of 2008
```

With this solution, the cell that prints the results shows:

```text
number of years:               34.0
(a) average annual log return: 7.583 %
(b) value of $1.00 at the end: $ 13.17
    check with simple returns: $ 13.17
```

</details>

### Exercise 1.4

Consider the daily log returns of American Express stock from January 1999 to December 2008 as in Exercise 1.1. Use the 5% significance level to perform the following tests: (a) Test the null hypothesis that the skewness measure of the returns is zero. (b) Test the null hypothesis that the excess kurtosis of the returns is zero.

**Task 28.** Solve Exercise 1.4 using the percentage log returns `r_axp_pct` from Exercise 1.1. Store the pairs `(t, p)` in `skew_test_axp` and `kurt_test_axp`.

In [ ]:
# Task 28: write your code in place of each `raise NotImplementedError(...)` line
skew_test_axp = None     # (a)
kurt_test_axp = None     # (b)

In [ ]:
# Tests for Task 28: run this cell. Every line should say PASS.
check_value('Ex. 1.4(a): (t, p)', skew_test_axp, expected=(-6.890755927, 5.549782855e-12))
check_value('Ex. 1.4(b): (t, p)', kurt_test_axp, expected=(66.43947934, 0.0))

In [ ]:
if skew_test_axp is None or kurt_test_axp is None:
    print("Complete the task above first.")
else:
    T = len(r_axp_pct)
    print("(a) skewness =", round(sample_skewness(r_axp_pct), 4), "  standard error sqrt(6/T) =", round((6 / T) ** 0.5, 4))
    print("    t =", round(skew_test_axp[0], 3), "  p =", skew_test_axp[1])
    print("(b) excess kurtosis =", round(excess_kurtosis(r_axp_pct), 4), "  standard error sqrt(24/T) =", round((24 / T) ** 0.5, 4))
    print("    t =", round(kurt_test_axp[0], 3), "  p =", kurt_test_axp[1])

<details>
<summary><b>Show solution to Task 28</b></summary>

```python
skew_test_axp = skewness_test(r_axp_pct)     # (a)
kurt_test_axp = kurtosis_test(r_axp_pct)     # (b)
```

With this solution, the cell that prints the results shows:

```text
(a) skewness = -0.3366   standard error sqrt(6/T) = 0.0488
    t = -6.891   p = 5.5497828554962325e-12
(b) excess kurtosis = 6.4903   standard error sqrt(24/T) = 0.0977
    t = 66.439   p = 0.0
```

**Conclusion.** In both tests $|t| > 1.96$, so both null hypotheses are rejected at the 5% level. Over 1999 to 2008 the daily log returns of AXP are negatively skewed and, above all, heavy-tailed: the excess kurtosis is about 6.5 with $t \approx 66$.

</details>

### Exercise 1.5

Daily foreign exchange rates (spot rates) are the noon buying rates in New York City certified by the Federal Reserve Bank of New York. Consider the exchange rates between the U.S. dollar and the Canadian dollar, the euro, the U.K. pound and the Japanese yen from January 4, 2000, to March 27, 2009.

(a) Compute the daily log return of each exchange rate.

(b) Compute the sample mean, standard deviation, skewness, excess kurtosis, minimum and maximum of the log returns of each exchange rate.

(c) Discuss the empirical characteristics of the log returns of exchange rates.

(d) Obtain a density plot of the daily log returns of the dollar–euro exchange rate. *(This part is done in Part B, once we have a plotting library.)*

The four files contain the year, month, day and the rate. The rate columns are named `rate` in `d-caus.txt` (Canadian dollars per U.S. dollar) and `value` in the other three files: `d-useu.txt` (U.S. dollars per euro), `d-jpus.txt` (yen per U.S. dollar) and `d-usuk.txt` (U.S. dollars per pound).

**Task 29.** Solve parts (a) and (b) of Exercise 1.5. These are **levels** of exchange rates, not returns, so use your `log_returns` from Task 6. Store the percentage log returns in `r_ca_pct`, `r_eu_pct`, `r_jp_pct` and `r_uk_pct`.

In [ ]:
# Task 29: write your code in place of each `raise NotImplementedError(...)` line
rate_ca = load_data("d-caus.txt")["rate"]
rate_eu = load_data("d-useu.txt")["value"]
rate_jp = load_data("d-jpus.txt")["value"]
rate_uk = load_data("d-usuk.txt")["value"]

r_ca_pct = None
r_eu_pct = None
r_jp_pct = None
r_uk_pct = None

In [ ]:
# Tests for Task 29: run this cell. Every line should say PASS.
check(summary_statistics, r_ca_pct, expected={'n': 2322, 'mean': -0.006819042022, 'std': 0.5876476303, 'skew': -0.2389874603, 'ex_kurt': 8.250179001, 'min': -5.071598585, 'max': 3.806962494}, label='Ex. 1.5(b), Canadian dollar')
check(summary_statistics, r_eu_pct, expected={'n': 2322, 'mean': 0.01099042899, 'std': 0.6539431268, 'skew': 0.1259859454, 'ex_kurt': 2.807000427, 'min': -3.003100771, 'max': 4.6207922}, label='Ex. 1.5(b), euro')
check(summary_statistics, r_jp_pct, expected={'n': 2322, 'mean': -0.002185054283, 'std': 0.6632315293, 'skew': -0.6709054718, 'ex_kurt': 4.584145526, 'min': -5.215647959, 'max': 2.70836454}, label='Ex. 1.5(b), yen')

In [ ]:
print("Percentage daily log returns of exchange rates")
print_header()
print_row("CA", r_ca_pct)
print_row("EU", r_eu_pct)
print_row("JP", r_jp_pct)
print_row("UK", r_uk_pct)

<details>
<summary><b>Show solution to Task 29</b></summary>

```python
rate_ca = load_data("d-caus.txt")["rate"]
rate_eu = load_data("d-useu.txt")["value"]
rate_jp = load_data("d-jpus.txt")["value"]
rate_uk = load_data("d-usuk.txt")["value"]

r_ca_pct = to_percent(log_returns(rate_ca))
r_eu_pct = to_percent(log_returns(rate_eu))
r_jp_pct = to_percent(log_returns(rate_jp))
r_uk_pct = to_percent(log_returns(rate_uk))
```

With this solution, the cell that prints the results shows:

```text
Percentage daily log returns of exchange rates
            n      Mean   St.dev.  Skewness  Ex.kurt.    Minimum   Maximum   t-ratio  p-value
CA       2322   -0.0068    0.5876   -0.2390    8.2502    -5.0716    3.8070    -0.559   0.5761
EU       2322    0.0110    0.6539    0.1260    2.8070    -3.0031    4.6208     0.810   0.4180
JP       2322   -0.0022    0.6632   -0.6709    4.5841    -5.2156    2.7084    -0.159   0.8739
UK       2322   -0.0057    0.6177   -0.3950    7.0655    -4.9662    4.4349    -0.448   0.6539
```

**(c) Discussion.** The mean daily log returns are essentially zero; none is significantly different from zero. Daily volatility is about 0.6% to 0.7%, much lower than for the individual stocks of Exercise 1.1 (2% to 3%). All four series have clearly positive excess kurtosis, i.e. heavy tails: strongest for the Canadian dollar (about 8.3) and the pound (about 7.1), weakest for the euro (about 2.8). Skewness is mild for the euro and the Canadian dollar but clearly negative for the pound and especially the yen. With the yen quoted per U.S. dollar, large negative returns are sudden appreciations of the yen, as in the 2008 crisis; with the pound quoted in dollars, they are sharp falls of sterling.

</details>

In [ ]:
show_progress()

---
# Part B. Letting the libraries do the work

You have now written every formula of Chapter 1 yourself. In practice, analysts rely on four libraries:

| Library | Purpose |
|---|---|
| **NumPy** | arrays: lists of numbers on which arithmetic works element by element, without loops |
| **SciPy** (`scipy.stats`) | probability distributions and statistical tests |
| **pandas** | tables with named columns and a date index, and reading data files |
| **Matplotlib** | plots |

They are included in Anaconda and Google Colab; otherwise install them with `pip install numpy scipy pandas matplotlib`.

Each section below redoes earlier work with a library and compares the result with **your** function using `check`. If the two disagree, either your function has a bug or the library uses a different definition. Telling these two cases apart is an important skill.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

### B.1 NumPy arrays: returns without loops

A NumPy array behaves like a list, but arithmetic is applied to every element at once. If `P` is an array of prices $P_0, \dots, P_T$, then `P[1:]` contains $P_1, \dots, P_T$ and `P[:-1]` contains $P_0, \dots, P_{T-1}$, so `P[1:] / P[:-1] - 1` evaluates Eq. (1.2) for all $t$ in one line.

In [ ]:
P = np.array([100.0, 110.0, 99.0, 120.0])

R = P[1:] / P[:-1] - 1          # Eq. (1.2) for every t
r = np.log(P[1:] / P[:-1])      # Eq. (1.6) for every t

print("simple returns R =", R)
print("log returns    r =", r)
print("1 + R_t[k] =", np.prod(1 + R), "  (product of the gross returns)")
print("r_t[k]     =", np.sum(r), "  (sum of the log returns)")

In [ ]:
check(simple_returns, [100.0, 110.0, 99.0, 120.0], expected=list(R), label="NumPy")
check(log_returns, [100.0, 110.0, 99.0, 120.0], expected=list(r), label="NumPy")
check(multiperiod_gross_return, list(R), expected=np.prod(1 + R), label="np.prod")
check(multiperiod_log_return, list(r), expected=np.sum(r), label="np.sum")

### B.2 Sample moments, and the `ddof` trap

`np.mean(x)` is Eq. (1.10). However, **`np.var` and `np.std` divide by $T$ by default**. The argument `ddof` ("delta degrees of freedom") changes the divisor to $T - \text{ddof}$, so Eq. (1.11) requires `ddof=1`. pandas, by contrast, uses `ddof=1` by default. Mixing the two is a common source of small but puzzling differences.

In [ ]:
x = np.array([2, 4, 4, 4, 5, 5, 7, 9])
print("np.var(x)          =", np.var(x), "   divides by T")
print("np.var(x, ddof=1)  =", np.var(x, ddof=1), "   divides by T - 1, as in Eq. (1.11)")
print("pd.Series(x).var() =", pd.Series(x).var(), "   pandas divides by T - 1 by default")

In [ ]:
x = np.array(sibm)        # IBM daily % simple returns from Section 1.2.1
check(sample_mean, sibm, expected=np.mean(x), label="np.mean")
check(sample_variance, sibm, expected=np.var(x, ddof=1), label="np.var with ddof=1")
check(sample_std, sibm, expected=np.std(x, ddof=1), label="np.std with ddof=1")

### B.3 Skewness and kurtosis: three conventions

Let $m_k = \frac{1}{T}\sum_t (x_t - \hat\mu_x)^k$ (divisor $T$). Three estimators of skewness and kurtosis are in common use:

| Estimator | Skewness | Kurtosis | Python |
|---|---|---|---|
| moment estimator | $g_1 = m_3 / m_2^{3/2}$ | $b_2 = m_4 / m_2^2$ | `stats.skew(x)`, `stats.kurtosis(x, fisher=False)` |
| adjusted estimator | $G_1 = g_1\sqrt{T(T-1)}\,/\,(T-2)$ | $G_2$ (excess) | `stats.skew(x, bias=False)`, `stats.kurtosis(x, bias=False)`, pandas `.skew()`, `.kurt()` |
| Tsay, Eqs. (1.12) and (1.13) | $\hat S = \sqrt{(T-1)/T}\; g_1$ | $\hat K = \frac{T-1}{T}\, b_2$ | your `sample_skewness`, `sample_kurtosis` |

Note also that `stats.kurtosis(x)` returns the **excess** kurtosis by default. All three estimators agree as $T$ grows, but in small samples they differ visibly. Compare them for the monthly GM returns ($T = 408$):

In [ ]:
R_gm = load_data("m-gm3dx7508.txt")["gm"]
x = 100 * np.array(R_gm)          # GM monthly % simple returns, as in Exercise 1.2
T = len(x)

g1 = stats.skew(x)
G1 = stats.skew(x, bias=False)
S_hat = ((T - 1) / T) ** 0.5 * g1
print("skewness:  g1 =", round(g1, 5), "  G1 =", round(G1, 5), "  Tsay =", round(S_hat, 5))

b2 = stats.kurtosis(x, fisher=False)
G2 = stats.kurtosis(x, bias=False)
K_hat = (T - 1) / T * b2
print("excess kurtosis:  b2 - 3 =", round(b2 - 3, 5), "  G2 =", round(G2, 5), "  Tsay =", round(K_hat - 3, 5))

In [ ]:
check(sample_skewness, list(x), expected=S_hat, label="Tsay's skewness from scipy")
check(sample_kurtosis, list(x), expected=K_hat, label="Tsay's kurtosis from scipy")

> **Remark 2.** The same word, "skewness", names three different estimators. When you try to reproduce a published table, first find out which definition the authors used.

### B.4 Distributions and tests with `scipy.stats`

Every distribution in `scipy.stats` provides `.pdf` (density), `.cdf` (distribution function), `.sf` (upper tail, $1 - \text{cdf}$), `.ppf` (quantile function) and `.logpdf`. The mean and standard deviation are set with `loc=` and `scale=`.

In [ ]:
print("Phi(1.96)                 =", stats.norm.cdf(1.96))
print("Z_0.025 (97.5% quantile)  =", stats.norm.ppf(0.975))
print("chi-squared(2) tail at 5  =", stats.chi2.sf(5, df=2), "  and exp(-5/2) =", np.exp(-2.5))

In [ ]:
check(normal_cdf, 1.3, expected=stats.norm.cdf(1.3), label="stats.norm.cdf")
check(two_sided_p_value, -1.7, expected=2 * stats.norm.sf(1.7), label="2 * stats.norm.sf")
check(normal_pdf, 0.4, 0.1, 2.0, expected=stats.norm.pdf(0.4, loc=0.1, scale=2.0), label="stats.norm.pdf")
check(cauchy_pdf, 2.5, expected=stats.cauchy.pdf(2.5), label="stats.cauchy.pdf")
check(normal_log_likelihood, [0.01, -0.02, 0.03], 0.0, 0.0004,
      expected=np.sum(stats.norm.logpdf([0.01, -0.02, 0.03], loc=0.0, scale=0.02)), label="stats.norm.logpdf")

The tests of Section 1.2.1 are also available. SciPy's `ttest_1samp` takes its p-value from the $t$ distribution, and its `jarque_bera` uses the moment estimators $g_1$ and $b_2$, so expect tiny differences from your functions for the IBM log returns:

In [ ]:
x = np.array(libm)                     # IBM daily % log returns
result = stats.ttest_1samp(x, 0.0)
print("zero-mean test:  t =", round(result.statistic, 4), "  p =", round(result.pvalue, 4))

result = stats.jarque_bera(x)
print("Jarque-Bera:     JB =", round(result.statistic, 1), "  p =", result.pvalue)

In [ ]:
t_scipy = stats.ttest_1samp(np.array(libm), 0.0).statistic
check(mean_t_test, libm, expected=(t_scipy, 2 * stats.norm.sf(abs(t_scipy))), label="stats.ttest_1samp")

### B.5 pandas: reading the data and working with whole tables

pandas can read the data files directly from the book's website with `pd.read_csv`. The option `sep=r"\s+"` says that columns are separated by one or more spaces, and `storage_options` sends a browser-like identification, which some websites require.

In [ ]:
url = DATA_URL + "d-3stocks9908.txt"
d3 = pd.read_csv(url, sep=r"\s+", storage_options={"User-Agent": "Mozilla/5.0"})

d3["date"] = pd.to_datetime(d3["date"].astype(str), format="%Y%m%d")    # 19990104 -> 1999-01-04
d3 = d3.set_index("date")
d3.head()

A **DataFrame** is a table: each column is a series and the dates form the index. Arithmetic applies to every column at once, so Exercise 1.1(b) and (c) take one line, and the summary table one more:

In [ ]:
r_pct = 100 * np.log(1 + d3)          # percentage log returns of all three stocks

r_pct.agg(["count", "mean", "std", "skew", "kurt", "min", "max"]).T.round(4)

The means, standard deviations, minima and maxima agree with your Exercise 1.1 table. The skewness and kurtosis differ in the later decimals because pandas uses the adjusted estimators $G_1$ and $G_2$ (Section B.3). To obtain Tsay's definitions, apply your own function to each column:

In [ ]:
table = {}
for name in r_pct.columns:
    table[name] = summary_statistics(list(r_pct[name]))
pd.DataFrame(table).T.round(4)

Exercise 1.1(d), the zero-mean test, is again a single line:

In [ ]:
t_ratio = r_pct.mean() / (r_pct.std() / np.sqrt(r_pct.count()))
p_value = 2 * stats.norm.sf(abs(t_ratio))
pd.DataFrame({"t-ratio": t_ratio, "p-value": p_value}).round(4)

The monthly data of Exercises 1.2 and 1.3 are read in the same way:

In [ ]:
url = DATA_URL + "m-gm3dx7508.txt"
dm = pd.read_csv(url, sep=r"\s+", storage_options={"User-Agent": "Mozilla/5.0"})
dm["date"] = pd.to_datetime(dm["date"].astype(str), format="%Y%m%d")
dm = dm.set_index("date")
dm.tail()

**Task 30.** Using the DataFrame `dm`, reproduce the table of Exercise 1.2(c): the summary statistics of the percentage log returns of GM, VW, EW and SP.

In [ ]:
# Task 30: write your code here


<details>
<summary><b>Show solution to Task 30</b></summary>

```python
r_pct_m = 100 * np.log(1 + dm)
r_pct_m.agg(["count", "mean", "std", "skew", "kurt", "min", "max"]).T.round(4)
```

As in the daily case, all columns except the skewness and kurtosis agree exactly with your Exercise 1.2 table.

</details>

**Task 31.** Redo Exercise 1.3 with pandas: compute the average annual log return of the S&P index and the final value of \$1.00.

In [ ]:
# Task 31: write your code here


<details>
<summary><b>Show solution to Task 31</b></summary>

```python
r_sp = np.log(1 + dm["sp"])
n = len(r_sp) / 12
print("average annual log return:", r_sp.sum() / n)
print("value of $1.00:", np.exp(r_sp.sum()), "  or, from simple returns:", (1 + dm["sp"]).prod())
```

About 7.58% per year, and \$13.17, as in Exercise 1.3.

</details>

### B.6 Plots with Matplotlib

**Figure 1.1 of the book**: the standard normal density, the mixture $0.95\,N(0,1) + 0.05\,N(0,16)$ and the Cauchy density.

In [ ]:
x = np.linspace(-4, 4, 401)                       # 401 points from -4 to 4
f_N = stats.norm.pdf(x)
f_mix = 0.95 * stats.norm.pdf(x, scale=1) + 0.05 * stats.norm.pdf(x, scale=4)
f_C = stats.cauchy.pdf(x)

plt.figure(figsize=(8, 4))
plt.plot(x, f_N, color="#2a78d6", linewidth=2, label="Standard normal")
plt.plot(x, f_mix, color="#eb6834", linewidth=2, label="Mixture 0.95 N(0,1) + 0.05 N(0,16)")
plt.plot(x, f_C, color="#1baf7a", linewidth=2, label="Cauchy")
plt.title("Figure 1.1: normal, mixture and Cauchy densities")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend(loc="upper left", frameon=False)
plt.grid(alpha=0.3)
plt.show()

**Time plots**, in the style of Figure 1.2: the monthly simple and log returns of GM. The two panels look almost the same, except that large negative returns are stretched further down in logs.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(dm.index, dm["gm"], color="#2a78d6", linewidth=1)
axes[0].set_ylabel("Simple return")
axes[0].set_title("Monthly returns of General Motors, 1975 to 2008")
axes[1].plot(dm.index, np.log(1 + dm["gm"]), color="#2a78d6", linewidth=1)
axes[1].set_ylabel("Log return")
axes[1].set_xlabel("Year")
plt.show()

**Empirical and normal densities**, in the style of Figure 1.4: a histogram of the GM monthly log returns together with the normal density that has the same mean and standard deviation. Note the higher peak and the heavier tails of the data.

In [ ]:
r_gm = np.log(1 + dm["gm"])
grid = np.linspace(r_gm.min(), r_gm.max(), 300)

plt.figure(figsize=(8, 4))
plt.hist(r_gm, bins=40, density=True, color="#2a78d6", alpha=0.35, edgecolor="white", label="Data (histogram)")
plt.plot(grid, stats.norm.pdf(grid, loc=r_gm.mean(), scale=r_gm.std()), "--", color="#eb6834",
         linewidth=2, label="Normal density, same mean and s.d.")
plt.title("GM monthly log returns and the normal distribution")
plt.xlabel("Log return")
plt.ylabel("Density")
plt.legend(frameon=False)
plt.show()

**Task 32.** Exercise 1.5(d): obtain a density plot of the daily log returns of the U.S. dollar–euro exchange rate (`d-useu.txt`), and add the normal density with the same mean and standard deviation. *Tip:* `stats.gaussian_kde(data)` returns a function that evaluates a smooth density estimate at the points of a grid.

In [ ]:
# Task 32: write your code here


<details>
<summary><b>Show solution to Task 32</b></summary>

```python
url = DATA_URL + "d-useu.txt"
eu = pd.read_csv(url, sep=r"\s+", storage_options={"User-Agent": "Mozilla/5.0"})
r_eu = 100 * np.log(eu["Value"]).diff().dropna()          # % daily log returns

grid = np.linspace(r_eu.min(), r_eu.max(), 400)
kde = stats.gaussian_kde(r_eu)

plt.figure(figsize=(8, 4))
plt.plot(grid, kde(grid), color="#2a78d6", linewidth=2, label="Density estimate")
plt.plot(grid, stats.norm.pdf(grid, loc=r_eu.mean(), scale=r_eu.std()), "--", color="#eb6834",
         linewidth=2, label="Normal density, same mean and s.d.")
plt.title("Daily log returns of the U.S. dollar-euro exchange rate, 2000 to 2009")
plt.xlabel("Log return (%)")
plt.ylabel("Density")
plt.legend(frameon=False)
plt.show()
```

`np.log(eu["Value"]).diff()` computes $p_t - p_{t-1}$, Eq. (1.6), for every day; `.dropna()` removes the first day, which has no previous price. The estimated density is taller and narrower in the centre than the normal density and has longer tails, the visual signature of the positive excess kurtosis (about 2.8) found in Exercise 1.5.

</details>

### B.7 Summary: your functions and their library equivalents

| Your function | Library equivalent | Note |
|---|---|---|
| `simple_returns(P)` | `P[1:] / P[:-1] - 1`; pandas `.pct_change()` | |
| `log_returns(P)` | `np.log(P[1:] / P[:-1])`; pandas `np.log(P).diff()` | |
| `multiperiod_gross_return(R)` | `np.prod(1 + R)`; pandas `(1 + R).prod()` | |
| `multiperiod_log_return(r)` | `np.sum(r)`; pandas `r.sum()` | |
| `sample_mean(x)` | `np.mean(x)`; pandas `.mean()` | |
| `sample_variance(x)`, `sample_std(x)` | `np.var(x, ddof=1)`, `np.std(x, ddof=1)`; pandas `.var()`, `.std()` | NumPy's default divides by $T$ |
| `sample_skewness(x)` | `stats.skew(x)`; pandas `.skew()` | different estimators (B.3) |
| `sample_kurtosis(x)`, `excess_kurtosis(x)` | `stats.kurtosis(x, fisher=False)`, `stats.kurtosis(x)`; pandas `.kurt()` | different estimators (B.3) |
| `normal_pdf`, `cauchy_pdf` | `stats.norm.pdf(x, loc, scale)`, `stats.cauchy.pdf(x)` | `scale` is the standard deviation |
| `normal_cdf`, `two_sided_p_value` | `stats.norm.cdf(z)`, `2 * stats.norm.sf(abs(t))` | |
| `mean_t_test` | `stats.ttest_1samp(x, 0)` | p-value from the $t$ distribution |
| `jarque_bera_test` | `stats.jarque_bera(x)` | uses $g_1$ and $b_2$ |
| `sample_mean_vector`, `sample_covariance_matrix` | `np.mean(X, axis=0)`, `np.cov(X, rowvar=False)`; pandas `.cov()` | |
| `normal_log_likelihood` | `np.sum(stats.norm.logpdf(r, mu, sigma))` | |
| `summary_statistics` | pandas `.agg([...])`, `.describe()` | |

A final check of the multivariate functions of Section 1.2.3:

In [ ]:
check(sample_mean_vector, X, expected=list(np.mean(np.array(X), axis=0)), label="np.mean(X, axis=0)")
check(sample_covariance_matrix, X, expected=np.cov(np.array(X), rowvar=False).tolist(), label="np.cov")

---
# Summary

How many tests have you passed?

In [ ]:
show_progress()

**Main points of Chapter 1**

1. **Simple and log returns.** Simple returns aggregate across assets through portfolio weights but compound multiplicatively over time; log returns add up over time but aggregate only approximately across assets. Since $\ln(1+R) \le R$, a log return is never larger than the corresponding simple return.
2. **Compounding.** Discrete compounding converges to continuous compounding, $A = Ce^{rn}$, and the present value is $C = Ae^{-rn}$. The arithmetic average overstates the compound growth rate, more so when returns are volatile.
3. **Sample moments.** The mean, variance (divisor $T-1$), skewness and kurtosis summarise the distribution of returns. Software packages use slightly different estimators, so always check the definition.
4. **Tests of normality.** $\hat S/\sqrt{6/T}$, $(\hat K - 3)/\sqrt{24/T}$ and the Jarque–Bera statistic, whose $\chi^2_2$ p-value is $e^{-JB/2}$.
5. **Empirical facts.** Daily mean returns are close to zero and hard to distinguish from zero; volatility dominates. Almost all return series have heavy tails (positive excess kurtosis), and log returns tend to be negatively skewed. These facts motivate the models in the rest of the book.

Chapter 2 asks whether past returns help to predict future returns, which leads to autocorrelation and linear time-series models.

# References

* Campbell, J. Y., Lo, A. W. and MacKinlay, A. C. (1997). *The Econometrics of Financial Markets*. Princeton, NJ: Princeton University Press.
* Jarque, C. M. and Bera, A. K. (1987). A test for normality of observations and regression residuals. *International Statistical Review*, 55, 163–172.
* Joanes, D. N. and Gill, C. A. (1998). Comparing measures of sample skewness and kurtosis. *Journal of the Royal Statistical Society, Series D (The Statistician)*, 47, 183–189.
* Snedecor, G. W. and Cochran, W. G. (1980). *Statistical Methods*, 7th edition. Ames, IA: Iowa State University Press.
* Tsay, R. S. (2010). *Analysis of Financial Time Series*, 3rd edition. Hoboken, NJ: Wiley.
* Data: R. S. Tsay, data sets for the 3rd edition, https://faculty.chicagobooth.edu/-/media/faculty/ruey-s-tsay/teaching/fts3/